# PHẦN 1: TỔNG QUAN ĐỒ ÁN & THIẾT LẬP MÔI TRƯỜNG

## Giới thiệu Đề tài: Xây dựng và Đánh giá Game AI Đa Cấp Độ (`Extended Tic-Tac-Toe M×N, K`)

* **Học phần:** Trí tuệ nhân tạo nâng cao (Advanced AI) —  `841458`
* **Sinh viên thực hiện:** **Trương Chí Long** — MSSV: `3123410198`
* **Giảng viên hướng dẫn:** **ThS. Đỗ Như Tài** — Khoa Công nghệ Thông tin, Trường Đại học Sài Gòn (SGU)
* **Kho lưu trữ & Thư mục làm việc:** `Advanced-AI-03 / Extended_Tic-Tac-Toe_Project`
* **Vai trò tài liệu:** *Báo cáo thực nghiệm tương tác và trình diễn hệ thống Game AI 7 cấp độ* (chạy trực tiếp trên Engine, Checkpoints đã kiểm định SHA-256 và Raw JSONL Logs thực nghiệm).

---

### 1.1. Bối cảnh bài toán `Extended Tic-Tac-Toe (M×N, K)`
Trên bàn cờ `3×3, K=3` cổ điển, không gian trạng thái chỉ có tối đa $3^9 = 19,683$ cấu hình ($\le 9!$). Các tác tử tìm kiếm toàn cây hoặc bộ luật heuristic chuẩn dễ kéo ván đấu về thế hòa, làm mờ ranh giới khác biệt giữa tìm kiếm đối kháng, học có giám sát và học tăng cường.

Dự án tổng quát hóa bài toán thành **Extended Tic-Tac-Toe `M×N, K`** (trò chơi đối kháng 2 người, quan sát đầy đủ, tất định, tổng bằng không *zero-sum*) và chọn **`4×4, K=4`** ($3^{16} \approx 4.3 \times 10^7$ cấu hình) làm **chuẩn thực nghiệm chính**, kết hợp các cấu hình đối chứng **`3×3, K=3`**, **`4×4, K=3`** và **`5×5, K=4`**.

### 1.2. Tóm tắt 3 Câu hỏi Nghiên cứu Cốt lõi (`RQ1 – RQ3`)

| Mã | Trọng tâm Nghiên cứu | Giả thuyết & Phương pháp Kiểm chứng |
|---|---|---|
| **RQ1** | **Hiệu quả của Hybrid MCTS (`PUCT + Policy/Value`) so với Pure MCTS (`UCT`) & Policy-only** | Trên bàn `4×4, K=4`, rollout ngẫu nhiên của MCTS thuần mất tín hiệu chiến thuật ở đầu/trung cuộc. Đưa xác suất tiên nghiệm $P(s,a)$ vào PUCT giúp hội tụ nhanh ở ngân sách thấp (`50 sims`), đồng thời **LRU Cache** giảm chi phí gọi PyTorch. |
| **RQ2** | **Biến thiên lợi thế đi trước ($\Delta\text{Score} = \text{Score}_X - \text{Score}_O$) theo cấp AI & luật `K`** | Nới bàn lên `4×4` nhưng giữ `K=3` khuếch đại cực mạnh lợi thế đi trước (`X`); tăng lên `4×4, K=4` mở rộng không gian phòng thủ, kéo tỷ lệ hòa lên cao và làm sụp đổ $\Delta\text{Score}$ về gần cân bằng. |
| **RQ3** | **Độ ổn định đa seed & tổng quát hóa: RL Self-Play (`DQN / TabularQ`) vs Supervised Learning** | Học có giám sát (*imitation learning*) có nguy cơ suy giảm mạnh khi gặp đối thủ tìm kiếm cây chưa từng thấy lúc huấn luyện (*Unseen Opponents*: `MCTS`, `AlphaBeta`), trong khi RL tự chơi với *Mixed Opponent Pool* qua 3 seed (`42, 123, 999`) tổng quát hóa tốt và ổn định hơn. |

In [1]:
# Khởi tạo môi trường, import thư viện khoa học, bổ sung src/ vào sys.path và seed_everything(42)
from __future__ import annotations

import json
import math
import platform
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

# Tự động dò tìm thư mục gốc dự án chứa `src/ttt_ai` (hoạt động từ Jupiter_Notebook/ hoặc root)
_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
PROJECT_ROOT: Path | None = next(
    (p for p in _candidates if (p / "src" / "ttt_ai").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Không tìm thấy thư mục `src/ttt_ai` từ vị trí hiện tại: {_cwd}"
    )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CONFIGS_DIR = PROJECT_ROOT / "configs"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
LOGS_DIR = PROJECT_ROOT / "logs"
REPORTS_DIR = PROJECT_ROOT / "reports"

from ttt_ai.config import BoardConfig, load_game_config, seed_everything

# Cố định toàn bộ PRNG (Python, NumPy, PyTorch, PYTHONHASHSEED) để tái lập 100%
GLOBAL_RNG = seed_everything(42, deterministic_torch=True)

# Cấu hình thẩm mỹ biểu đồ Matplotlib nhất quán với `ttt_ai.benchmark.visualizer`
PALETTE: dict[str, str] = {
    "blue": "#0072B2",
    "vermilion": "#D55E00",
    "green": "#009E73",
    "amber": "#E69F00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "charcoal": "#333333",
    "gray": "#7F8C8D",
}

PLOT_RC: dict[str, Any] = {
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.facecolor": "white",
    "axes.facecolor": "white",
}
plt.rcParams.update(PLOT_RC)

env_summary = pd.DataFrame(
    [
        {"Thành phần": "Project Root", "Giá trị": str(PROJECT_ROOT)},
        {"Thành phần": "Working Dir (CWD)", "Giá trị": str(_cwd)},
        {"Thành phần": "Python / Platform", "Giá trị": f"{platform.python_version()} ({platform.system()})"},
        {"Thành phần": "NumPy / Pandas", "Giá trị": f"{np.__version__} / {pd.__version__}"},
        {"Thành phần": "PyTorch / Device", "Giá trị": f"{torch.__version__} / {'cuda' if torch.cuda.is_available() else 'cpu'}"},
        {"Thành phần": "Matplotlib Backend", "Giá trị": f"{matplotlib.__version__} ({matplotlib.get_backend()})"},
        {"Thành phần": "YAML (PyYAML)", "Giá trị": yaml.__version__},
        {"Thành phần": "Checkpoints (.pt/.pkl)", "Giá trị": str(len(list(CHECKPOINTS_DIR.glob('*.pt'))) + len(list(CHECKPOINTS_DIR.glob('*.pkl'))))},
        {"Thành phần": "Raw Logs (.jsonl)", "Giá trị": str(len(list(LOGS_DIR.glob('*.jsonl'))))},
        {"Thành phần": "Global Seed", "Giá trị": "42 (deterministic_torch=True)"},
    ]
)
print("Khởi tạo môi trường thành công & đã cố định seed_everything(42).")
display(env_summary)

Khởi tạo môi trường thành công & đã cố định seed_everything(42).


,Thành phần,Giá trị
0,Project Root,C:\Users\Ron\Documents\GitHub\Advanced-AI-03\E...
1,Working Dir (CWD),C:\Users\Ron\Documents\GitHub\Advanced-AI-03\E...
2,Python / Platform,3.14.7 (Windows)
3,NumPy / Pandas,2.5.4 / 3.0.6
4,PyTorch / Device,2.15.0+cpu / cpu
5,Matplotlib Backend,3.11.2 (Agg)
6,YAML (PyYAML),6.0.3
7,Checkpoints (.pt/.pkl),15
8,Raw Logs (.jsonl),24
9,Global Seed,42 (deterministic_torch=True)


---
# PHẦN 2: LÕI TRÒ CHƠI & GIAO DIỆN HỢP ĐỒNG ENGINE

## Mô hình hóa MDP & Khởi tạo `ExtendedTicTacToe` (`4×4, K=4` và `3×3, K=3`)

Engine trò chơi nằm hoàn toàn độc lập trong phân hệ `ttt_ai.engine` (thuần Python + NumPy, không phụ thuộc PyTorch hay UI).

* **Không gian trạng thái (`State`):** Ma trận `board` kích thước `(M, N)` nhận giá trị `{-1 (O), 0 (Trống), +1 (X)}` cùng lượt chơi hiện tại `current_player ∈ {+1, -1}`.
* **Không gian hành động (`Action`):** Chỉ số ô phẳng `a ∈ {0, 1, ..., M*N - 1}` theo thứ tự hàng-cột `(row, col) = divmod(a, N)`.
* **Hợp đồng 7+1 Game API thống nhất:**
  1. `reset()` — Xóa bàn cờ, đặt lượt đầu cho `Player.X (+1)`.
  2. `legal_actions()` — Trả về danh sách chỉ số ô trống hợp lệ (hoặc `[]` nếu kết thúc).
  3. `step(action)` — Kiểm tra chặt tính hợp lệ (ném `IllegalMoveError` nếu sai), cập nhật bàn cờ, kiểm thắng $O(K)$ theo 4 trục.
  4. `is_terminal()` — Kiểm tra ván đấu đã thắng hoặc hòa đầy bàn chưa.
  5. `result(player)` — Payoff zero-sum `+1.0` (thắng), `-1.0` (thua), `0.0` (hòa), `None` (chưa kết thúc).
  6. `render()` — Kết xuất bàn cờ ASCII kèm tọa độ và đánh dấu đường thắng.
  7. `clone()` & `undo()` — Sao chép độc lập và hoàn tác $O(1)$ phục vụ tìm kiếm cây Alpha-Beta / MCTS.

In [3]:
# Nạp YAML config & khởi tạo ExtendedTicTacToe trên 4x4 K=4 (chính) và 3x3 K=3 (đối chứng)
from ttt_ai.engine import (
    EMPTY,
    PLAYER_O,
    PLAYER_X,
    ExtendedTicTacToe,
    IllegalMoveError,
    Player,
    action_to_coords,
    check_win_around,
    coords_to_action,
    positional_weights,
    precompute_win_windows,
)

# Đọc trực tiếp file YAML cấu hình & kiểm định qua dataclass GameConfig
with (CONFIGS_DIR / "game_4x4_k4.yaml").open("r", encoding="utf-8") as f:
    raw_yaml_4x4 = yaml.safe_load(f)

cfg_4x4 = load_game_config(CONFIGS_DIR / "game_4x4_k4.yaml")
cfg_3x3 = load_game_config(CONFIGS_DIR / "game_3x3_k3.yaml")

game_4x4 = ExtendedTicTacToe(
    rows=cfg_4x4.board.rows,
    cols=cfg_4x4.board.cols,
    win_k=cfg_4x4.board.win_k,
)
game_3x3 = ExtendedTicTacToe(
    rows=cfg_3x3.board.rows,
    cols=cfg_3x3.board.cols,
    win_k=cfg_3x3.board.win_k,
)

windows_4x4 = precompute_win_windows(4, 4, 4)
windows_3x3 = precompute_win_windows(3, 3, 3)

print("=== Cấu hình YAML chuẩn thực nghiệm chính (configs/game_4x4_k4.yaml) ===")
print(json.dumps(raw_yaml_4x4, indent=2, ensure_ascii=False))
print(f"\n4x4, K=4 -> Số ô: {game_4x4.num_cells} | Số đường thắng K=4: {len(windows_4x4)}")
print("Ma trận trọng số vị trí positional_weights(4, 4, 4):")
print(positional_weights(4, 4, 4))

print(f"\n3x3, K=3 -> Số ô: {game_3x3.num_cells} | Số đường thắng K=3: {len(windows_3x3)}")
print("Ma trận trọng số vị trí positional_weights(3, 3, 3):")
print(positional_weights(3, 3, 3))

=== Cấu hình YAML chuẩn thực nghiệm chính (configs/game_4x4_k4.yaml) ===
{
  "name": "game_4x4_k4",
  "description": "Primary Benchmark (4x4 board, K=4 to win) for RQ3, RQ4, RQ5",
  "board": {
    "rows": 4,
    "cols": 4,
    "win_k": 4
  },
  "seed": 42,
  "default_time_budget_ms": 50.0,
  "default_n_simulations": 200
}

4x4, K=4 -> Số ô: 16 | Số đường thắng K=4: 10
Ma trận trọng số vị trí positional_weights(4, 4, 4):
[[3 2 2 3]
 [2 3 3 2]
 [2 3 3 2]
 [3 2 2 3]]

3x3, K=3 -> Số ô: 9 | Số đường thắng K=3: 8
Ma trận trọng số vị trí positional_weights(3, 3, 3):
[[3 2 3]
 [2 4 2]
 [3 2 3]]


In [4]:
# Kiểm chứng Game API: step(), undo(), clone(), check_win_around O(K), canonical_board(), render()
game_4x4.reset()

# Chuỗi nước đi mẫu giúp Player X thắng theo đường chéo chính: (0,0)->(1,1)->(2,2)->(3,3)
demo_moves = [0, 1, 5, 2, 10, 3]
for mv in demo_moves:
    _, reward, done, info = game_4x4.step(mv)

print(f"Sau {game_4x4.move_count} nước đi:")
print(f"  - Lượt hiện tại: Player {game_4x4.current_player.symbol} ({int(game_4x4.current_player):+d})")
print(f"  - Các nước hợp lệ còn lại ({len(game_4x4.legal_actions())}): {game_4x4.legal_actions()}")
print(f"  - Đã kết thúc? {game_4x4.is_terminal()} | Result(X) = {game_4x4.result(Player.X)}")

# 1. Minh họa canonical_board() khi đang là lượt X (+1) và khi thử chuyển sang lượt O (-1)
print("\n[Canonical Board tại lượt X (+1)] (Quân mình = +1, Đối thủ = -1):")
print(game_4x4.canonical_board())

clone_for_o = game_4x4.clone()
clone_for_o.step(12)  # X đánh ô (3,0), chuyển sang lượt O
print(f"\n[Canonical Board trên bản clone khi tới lượt O (-1)] (Quân O biến thành +1, Quân X thành -1):")
print(clone_for_o.canonical_board())
print(f"Kiểm chứng tính cô lập của clone(): game_4x4.move_count vẫn là {game_4x4.move_count} (clone là {clone_for_o.move_count}).")

# 2. Thực hiện nước thắng quyết định tại ô #15 = (3, 3) và kiểm tra check_win_around O(K)
board_after, reward, done, info = game_4x4.step(15)
row_15, col_15 = action_to_coords(15, game_4x4.cols)
win_coords = check_win_around(board_after, int(row_15), int(col_15), int(Player.X), game_4x4.win_k)

print(f"\nSau khi X đánh nước quyết định #{15} -> (row={row_15}, col={col_15}):")
print(f"  - reward={reward}, done={done}, winner=Player {game_4x4.winner.symbol}")
print(f"  - Result(X)={game_4x4.result(Player.X)}, Result(O)={game_4x4.result(Player.O)}")
print(f"  - check_win_around O(K) trả về: {win_coords}")
print("\n" + game_4x4.render())

# 3. Kiểm thử xử lý ngoại lệ IllegalMoveError (đánh sau khi kết thúc & đánh vào ô đã chiếm)
try:
    game_4x4.step(4)
except IllegalMoveError as exc:
    print(f"\n[Kiểm thử An toàn API #1] Bắt đúng IllegalMoveError khi ván đã kết thúc: {exc}")

# 4. Kiểm thử undo() hoàn tác O(1) từ trạng thái terminal về trạng thái trước đó
game_4x4.undo()
print(f"[undo() O(1)] Đã hoàn tác nước thắng: is_terminal={game_4x4.is_terminal()}, "
      f"move_count={game_4x4.move_count}, turn=Player {game_4x4.current_player.symbol}")

try:
    game_4x4.step(0)  # Ô (0,0) đã có quân X
except IllegalMoveError as exc:
    print(f"[Kiểm thử An toàn API #2] Bắt đúng IllegalMoveError khi đánh vào ô đã chiếm: {exc}")

Sau 6 nước đi:
  - Lượt hiện tại: Player X (+1)
  - Các nước hợp lệ còn lại (10): [4, 6, 7, 8, 9, 11, 12, 13, 14, 15]
  - Đã kết thúc? False | Result(X) = None

[Canonical Board tại lượt X (+1)] (Quân mình = +1, Đối thủ = -1):
[[ 1 -1 -1 -1]
 [ 0  1  0  0]
 [ 0  0  1  0]
 [ 0  0  0  0]]

[Canonical Board trên bản clone khi tới lượt O (-1)] (Quân O biến thành +1, Quân X thành -1):
[[-1  1  1  1]
 [ 0 -1  0  0]
 [ 0  0 -1  0]
 [-1  0  0  0]]
Kiểm chứng tính cô lập của clone(): game_4x4.move_count vẫn là 6 (clone là 7).

Sau khi X đánh nước quyết định #15 -> (row=3, col=3):
  - reward=1.0, done=True, winner=Player X
  - Result(X)=1.0, Result(O)=-1.0
  - check_win_around O(K) trả về: [(0, 0), (1, 1), (2, 2), (3, 3)]

Board: 4x4 (K=4) | Result: Player X wins!
     0   1   2   3 
   +---+---+---+---+
 0 |[X]| O | O | O |
   +---+---+---+---+
 1 | . |[X]| . | . |
   +---+---+---+---+
 2 | . | . |[X]| . |
   +---+---+---+---+
 3 | . | . | . |[X]|
   +---+---+---+---+
Winning line: [(0, 0), (1,

---
# PHẦN 3: THI TRIỂN & ĐỐI ĐẦU 7 CẤP ĐỘ TÁC TỬ AI

## Nguyên lý Phân tầng 7 Cấp độ Tác tử AI

Mọi tác tử từ **Cấp 1 đến Cấp 7** đều kế thừa hợp đồng `BaseAgent`, tương tác với `ExtendedTicTacToe` thuần qua Game API công khai (được kiểm chứng tự động bằng AST test không chạm biến nội bộ `_board`/`_history`) và trả về `AgentDecision` chuẩn hóa gồm: `action`, `latency_ms`, `nodes_visited`, `simulations`, `depth_reached`, `value_estimate`, `action_scores`, và `prior_probs`.

| Cấp | Tác tử (`ttt_ai.agents`) | Nhóm Phương pháp | Đặc điểm Kỹ thuật Cốt lõi |
|---:|---|---|---|
| **1** | `RandomAgent` | Baseline | Chọn đều ngẫu nhiên từ `legal_actions()` qua `np.random.Generator` cô lập theo `seed`. |
| **2** | `RuleBasedAgent` | Heuristic / Luật | Tổng quát hóa 6 tầng ưu tiên **Newell & Simon** cho `M×N, K`: `IMMEDIATE_WIN` → `IMMEDIATE_BLOCK` → `CREATE_FORK` → `BLOCK_FORK` → `POSITIONAL` → `FALLBACK`. |
| **3** | `AlphaBetaAgent` (`MinimaxAgent`) | Tìm kiếm Đối kháng | Negamax + `step()/undo()` trên 1 bản `clone()`, lượng giá `evaluate_board` ($\tanh$ trên cửa sổ mở $10^{c-1}$), **Alpha-Beta Pruning**, **Move Ordering**, và **Iterative Deepening** theo `time_budget_ms`. |
| **4** | `MCTSAgent` | Monte Carlo Tree Search | **4 pha UCT** (`Selection`, `Expansion`, `Random Rollout`, `Backpropagation`), giải phóng cây sau mỗi nước (`__slots__`), hỗ trợ giới hạn `n_simulations` hoặc `time_budget_ms`. |
| **5** | `PolicyNetworkAgent` / `ValueGreedyAgent` | Học có Giám sát (SL) | CNN+MLP `(B, 1, M, N)`: `FastPolicyNet`, `StrongPolicyNet` và `ValueNet`. Áp dụng **Action Masking (`-1e9`)** loại bỏ hoàn toàn nước đi bất hợp lệ. |
| **6** | `DQNAgent` / `TabularQAgent` | Học Tăng cường (RL) | `TabularQAgent` (bảng Q cho `3×3 K=3`) và `DQNAgent` (*Double DQN + Replay Buffer 50,000 + Mixed Pool 70% Self-Play / 30% Baseline* cho `4×4`). |
| **7** | `HybridMCTSAgent` *(AlphaGo-inspired)* | Lai Cây Tìm kiếm + DL | Kết hợp cây **PUCT** với `StrongPolicyNet` cung cấp prior $P(s,a)$, **LRU Cache** theo hash bàn cờ, và 2 chế độ lượng giá lá (`rollout` thuần hoặc `mixed` với `ValueNet`). |

> **Minh bạch học thuật:** Cấp 7 (`HybridMCTSAgent`) được định vị đúng phạm vi **AlphaGo-inspired** (Supervised Policy/Value + PUCT trên CPU), không phóng đại thành kiến trúc AlphaZero phân tán đa GPU/TPU.

In [5]:
# Cấp 1 (RandomAgent) & Cấp 2 (RuleBasedAgent 6 tầng luật): Ván đấu mẫu & đo latency_ms
from ttt_ai.agents import AgentDecision, BaseAgent, RandomAgent, RuleBasedAgent


def play_demo_match(
    agent_x: BaseAgent,
    agent_o: BaseAgent,
    *,
    rows: int = 4,
    cols: int = 4,
    win_k: int = 4,
    show_board: bool = True,
    verbose_moves: bool = False,
) -> dict[str, Any]:
    """Chạy một ván đấu trọn vẹn qua public Game API và thu thập telemetry từng bên."""
    env = ExtendedTicTacToe(rows=rows, cols=cols, win_k=win_k)
    env.reset()

    stats: dict[str, dict[str, list[float]]] = {
        "X": {"lat": [], "nodes": [], "sims": [], "depth": []},
        "O": {"lat": [], "nodes": [], "sims": [], "depth": []},
    }
    trace: list[str] = []

    while not env.is_terminal():
        player = env.current_player
        active_agent = agent_x if player is Player.X else agent_o
        decision: AgentDecision = active_agent.select_action(env)

        r, c = action_to_coords(decision.action, env.cols)
        trace.append(f"{env.move_count + 1}.{player.symbol}({r},{c})[#{decision.action}]")

        s_key = player.symbol
        stats[s_key]["lat"].append(decision.latency_ms)
        stats[s_key]["nodes"].append(float(decision.nodes_visited))
        stats[s_key]["sims"].append(float(decision.simulations))
        stats[s_key]["depth"].append(float(decision.depth_reached))

        if verbose_moves:
            print(
                f"  Ply {env.move_count + 1:02d} | {player.symbol} ({active_agent.name}) -> "
                f"({r},{c})[#{decision.action}] | {decision.latency_ms:.3f} ms | "
                f"nodes={decision.nodes_visited}, sims={decision.simulations}, depth={decision.depth_reached}"
            )
        env.step(decision.action)

    res_x = env.result(Player.X)
    winner_label = (
        f"{agent_x.name} (X)" if res_x == 1.0 else (f"{agent_o.name} (O)" if res_x == -1.0 else "Draw")
    )
    if show_board:
        print(f"Kết quả: {winner_label} | Số nước: {env.move_count}")
        print(f"Chuỗi nước: {' -> '.join(trace)}")
        print(env.render())

    def _agg(d: dict[str, list[float]]) -> dict[str, float]:
        return {
            "mean_ms": float(np.mean(d["lat"])) if d["lat"] else 0.0,
            "max_ms": float(np.max(d["lat"])) if d["lat"] else 0.0,
            "avg_nodes": float(np.mean(d["nodes"])) if d["nodes"] else 0.0,
            "avg_sims": float(np.mean(d["sims"])) if d["sims"] else 0.0,
            "avg_depth": float(np.mean(d["depth"])) if d["depth"] else 0.0,
        }

    return {
        "winner": winner_label,
        "score_x": 1.0 if res_x == 1.0 else (0.5 if res_x == 0.0 else 0.0),
        "score_o": 1.0 if res_x == -1.0 else (0.5 if res_x == 0.0 else 0.0),
        "num_moves": env.move_count,
        "X": _agg(stats["X"]),
        "O": _agg(stats["O"]),
        "game": env,
    }


l1_random = RandomAgent(name="L1_Random", seed=42)
l2_rule = RuleBasedAgent(name="L2_RuleBased", seed=42, tie_break="random")

print("=== Ván đấu mẫu: L1_Random (X) vs L2_RuleBased (O) trên 4x4 K=4 ===")
sample_l1_l2 = play_demo_match(l1_random, l2_rule, verbose_moves=True, show_board=True)

# Đối đầu cân bằng 10 ván (5 ván X, 5 ván O) để đo latency_ms & tỷ số thực tế
rule_scores: list[float] = []
rand_lats: list[float] = []
rule_lats: list[float] = []
w_d_l = [0, 0, 0]

for idx in range(10):
    l1_random.reset(seed=100 + idx)
    l2_rule.reset(seed=200 + idx)
    if idx % 2 == 0:
        out = play_demo_match(l2_rule, l1_random, show_board=False)
        s_rule = out["score_x"]
        rule_lats.append(out["X"]["mean_ms"])
        rand_lats.append(out["O"]["mean_ms"])
    else:
        out = play_demo_match(l1_random, l2_rule, show_board=False)
        s_rule = out["score_o"]
        rand_lats.append(out["X"]["mean_ms"])
        rule_lats.append(out["O"]["mean_ms"])
    rule_scores.append(s_rule)
    if s_rule == 1.0:
        w_d_l[0] += 1
    elif s_rule == 0.5:
        w_d_l[1] += 1
    else:
        w_d_l[2] += 1

df_l1_l2 = pd.DataFrame(
    [
        {"Agent": "L2_RuleBased (6 tầng luật)", "W-D-L": f"{w_d_l[0]}-{w_d_l[1]}-{w_d_l[2]}", "Score": float(np.mean(rule_scores)), "Mean Latency (ms)": round(float(np.mean(rule_lats)), 4)},
        {"Agent": "L1_Random (Baseline)", "W-D-L": f"{w_d_l[2]}-{w_d_l[1]}-{w_d_l[0]}", "Score": 1.0 - float(np.mean(rule_scores)), "Mean Latency (ms)": round(float(np.mean(rand_lats)), 4)},
    ]
)
print("\n=== Kết quả đối đầu cân bằng 10 ván (5 X / 5 O) trên 4x4 K=4 ===")
display(df_l1_l2)

=== Ván đấu mẫu: L1_Random (X) vs L2_RuleBased (O) trên 4x4 K=4 ===
  Ply 01 | X (L1_Random) -> (0,1)[#1] | 0.112 ms | nodes=1, sims=0, depth=0
  Ply 02 | O (L2_RuleBased) -> (1,2)[#6] | 0.153 ms | nodes=45, sims=0, depth=1
  Ply 03 | X (L1_Random) -> (3,0)[#12] | 0.032 ms | nodes=1, sims=0, depth=0
  Ply 04 | O (L2_RuleBased) -> (2,2)[#10] | 0.077 ms | nodes=39, sims=0, depth=1
  Ply 05 | X (L1_Random) -> (2,1)[#9] | 0.025 ms | nodes=1, sims=0, depth=0
  Ply 06 | O (L2_RuleBased) -> (1,1)[#5] | 0.066 ms | nodes=33, sims=0, depth=1
  Ply 07 | X (L1_Random) -> (1,3)[#7] | 0.022 ms | nodes=1, sims=0, depth=0
  Ply 08 | O (L2_RuleBased) -> (3,3)[#15] | 0.056 ms | nodes=27, sims=0, depth=1
  Ply 09 | X (L1_Random) -> (1,0)[#4] | 0.022 ms | nodes=1, sims=0, depth=0
  Ply 10 | O (L2_RuleBased) -> (0,0)[#0] | 0.052 ms | nodes=21, sims=0, depth=1
Kết quả: L2_RuleBased (O) | Số nước: 10
Chuỗi nước: 1.X(0,1)[#1] -> 2.O(1,2)[#6] -> 3.X(3,0)[#12] -> 4.O(2,2)[#10] -> 5.X(2,1)[#9] -> 6.O(1,1)[#5] ->

,Agent,W-D-L,Score,Mean Latency (ms)
0,L2_RuleBased (6 tầng luật),9-1-0,0.95,0.0782
1,L1_Random (Baseline),0-1-9,0.05,0.0263


In [6]:
# Cấp 3 (AlphaBetaAgent: Heuristic + Iterative Deepening) & Cấp 4 (MCTSAgent: UCT 4 pha)
from ttt_ai.agents import AlphaBetaAgent, MCTSAgent, evaluate_board

l3_alphabeta = AlphaBetaAgent(
    name="L3_AlphaBeta",
    max_depth=4,
    use_alpha_beta=True,
    use_move_ordering=True,
    iterative_deepening=True,
    time_budget_ms=50.0,
    seed=42,
)
l4_mcts = MCTSAgent(
    name="L4_MCTS",
    n_simulations=200,
    c_uct=math.sqrt(2.0),
    time_budget_ms=50.0,
    seed=42,
)

# Minh họa hàm lượng giá heuristic evaluate_board() trên một thế cờ đang diễn ra
probe_game = ExtendedTicTacToe(4, 4, 4)
for a in [5, 6, 9]:  # X có (1,1) và (2,1); O có (1,2)
    probe_game.step(a)
print(
    f"Kiểm chứng evaluate_board() trên thế cờ trung cuộc (3 ply, tới lượt O): "
    f"V(O) = {evaluate_board(probe_game, Player.O):+.4f}, V(X) = {evaluate_board(probe_game, Player.X):+.4f}"
)

print("\n=== Trận 1: L3_AlphaBeta (X) vs L4_MCTS (O) [budget=50 ms/move] ===")
l3_alphabeta.reset(seed=42)
l4_mcts.reset(seed=42)
match_ab_vs_mcts = play_demo_match(l3_alphabeta, l4_mcts, show_board=True)

print("\n=== Trận 2 (Đảo lượt): L4_MCTS (X) vs L3_AlphaBeta (O) [budget=50 ms/move] ===")
l3_alphabeta.reset(seed=43)
l4_mcts.reset(seed=43)
match_mcts_vs_ab = play_demo_match(l4_mcts, l3_alphabeta, show_board=True)

telemetry_l3_l4 = pd.DataFrame(
    [
        {
            "Trận": "L3_AlphaBeta (X) vs L4_MCTS (O)",
            "Kết quả": match_ab_vs_mcts["winner"],
            "Plies": match_ab_vs_mcts["num_moves"],
            "X Mean/Max (ms)": f"{match_ab_vs_mcts['X']['mean_ms']:.2f} / {match_ab_vs_mcts['X']['max_ms']:.2f}",
            "X Nodes / Sims / Depth": f"{match_ab_vs_mcts['X']['avg_nodes']:.1f} / {match_ab_vs_mcts['X']['avg_sims']:.0f} / {match_ab_vs_mcts['X']['avg_depth']:.2f}",
            "O Mean/Max (ms)": f"{match_ab_vs_mcts['O']['mean_ms']:.2f} / {match_ab_vs_mcts['O']['max_ms']:.2f}",
            "O Nodes / Sims / Depth": f"{match_ab_vs_mcts['O']['avg_nodes']:.1f} / {match_ab_vs_mcts['O']['avg_sims']:.0f} / {match_ab_vs_mcts['O']['avg_depth']:.2f}",
        },
        {
            "Trận": "L4_MCTS (X) vs L3_AlphaBeta (O)",
            "Kết quả": match_mcts_vs_ab["winner"],
            "Plies": match_mcts_vs_ab["num_moves"],
            "X Mean/Max (ms)": f"{match_mcts_vs_ab['X']['mean_ms']:.2f} / {match_mcts_vs_ab['X']['max_ms']:.2f}",
            "X Nodes / Sims / Depth": f"{match_mcts_vs_ab['X']['avg_nodes']:.1f} / {match_mcts_vs_ab['X']['avg_sims']:.0f} / {match_mcts_vs_ab['X']['avg_depth']:.2f}",
            "O Mean/Max (ms)": f"{match_mcts_vs_ab['O']['mean_ms']:.2f} / {match_mcts_vs_ab['O']['max_ms']:.2f}",
            "O Nodes / Sims / Depth": f"{match_mcts_vs_ab['O']['avg_nodes']:.1f} / {match_mcts_vs_ab['O']['avg_sims']:.0f} / {match_mcts_vs_ab['O']['avg_depth']:.2f}",
        },
    ]
)
display(telemetry_l3_l4)

Kiểm chứng evaluate_board() trên thế cờ trung cuộc (3 ply, tới lượt O): V(O) = -0.0110, V(X) = +0.0110

=== Trận 1: L3_AlphaBeta (X) vs L4_MCTS (O) [budget=50 ms/move] ===
Kết quả: Draw | Số nước: 16
Chuỗi nước: 1.X(0,0)[#0] -> 2.O(3,3)[#15] -> 3.X(0,3)[#3] -> 4.O(2,1)[#9] -> 5.X(3,1)[#13] -> 6.O(0,2)[#2] -> 7.X(1,2)[#6] -> 8.O(2,0)[#8] -> 9.X(0,1)[#1] -> 10.O(2,3)[#11] -> 11.X(2,2)[#10] -> 12.O(3,2)[#14] -> 13.X(1,0)[#4] -> 14.O(1,3)[#7] -> 15.X(1,1)[#5] -> 16.O(3,0)[#12]
Board: 4x4 (K=4) | Result: Draw
     0   1   2   3 
   +---+---+---+---+
 0 | X | X | O | X |
   +---+---+---+---+
 1 | X | X | X | O |
   +---+---+---+---+
 2 | O | O | X | O |
   +---+---+---+---+
 3 | O | X | O | O |
   +---+---+---+---+

=== Trận 2 (Đảo lượt): L4_MCTS (X) vs L3_AlphaBeta (O) [budget=50 ms/move] ===
Kết quả: Draw | Số nước: 16
Chuỗi nước: 1.X(0,0)[#0] -> 2.O(1,1)[#5] -> 3.X(2,3)[#11] -> 4.O(2,1)[#9] -> 5.X(2,0)[#8] -> 6.O(0,3)[#3] -> 7.X(3,0)[#12] -> 8.O(1,0)[#4] -> 9.X(3,1)[#13] -> 10.O(0,2)[#2] 

,Trận,Kết quả,Plies,X Mean/Max (ms),X Nodes / Sims / Depth,O Mean/Max (ms),O Nodes / Sims / Depth
0,L3_AlphaBeta (X) vs L4_MCTS (O),Draw,16,11.99 / 31.24,1066.2 / 0 / 3.75,14.49 / 27.06,153.0 / 200 / 3.00
1,L4_MCTS (X) vs L3_AlphaBeta (O),Draw,16,15.99 / 29.80,158.2 / 200 / 3.50,8.36 / 21.33,803.2 / 0 / 3.50


In [8]:
# Kiểm định an toàn checkpoint (*.meta.json + SHA-256) & Khởi tạo Cấp 5 (SL) + Cấp 6 (RL)
from ttt_ai.agents import DQNAgent, PolicyNetworkAgent, TabularQAgent, ValueGreedyAgent
from ttt_ai.engine import CheckpointVerificationError
from ttt_ai.training.checkpoint_io import verify_checkpoint

board_4x4_cfg = BoardConfig(4, 4, 4)
board_3x3_cfg = BoardConfig(3, 3, 3)

ckpt_specs = [
    ("L5 FastPolicyNet", CHECKPOINTS_DIR / "fastpolicynet_4x4_k4.pt", board_4x4_cfg, "FastPolicyNet", "supervised"),
    ("L5 StrongPolicyNet", CHECKPOINTS_DIR / "strongpolicynet_4x4_k4.pt", board_4x4_cfg, "StrongPolicyNet", "supervised"),
    ("L5 ValueNet", CHECKPOINTS_DIR / "valuenet_4x4_k4.pt", board_4x4_cfg, "ValueNet", "supervised"),
    ("L6 DQNAgent (seed=42)", CHECKPOINTS_DIR / "dqnnet_4x4_k4_seed42.pt", board_4x4_cfg, "DQNNet", "rl_dqn"),
    ("L6 TabularQAgent (seed=42)", CHECKPOINTS_DIR / "tabular_q_3x3_k3_seed42.pkl", board_3x3_cfg, "TabularQAgent", "rl_tabular"),
]

verified_rows: list[dict[str, Any]] = []
for label, path, b_cfg, exp_cls, exp_type in ckpt_specs:
    meta = verify_checkpoint(
        path,
        board_config=b_cfg,
        expected_agent_class=exp_cls,
        expected_training_type=exp_type,
    )
    metric_note = (
        f"val_acc={meta.best_val_accuracy:.4f}"
        if meta.best_val_accuracy is not None
        else (
            f"val_loss={meta.best_val_loss:.4f}"
            if meta.best_val_loss is not None
            else f"eval_vs_rand={meta.final_eval_score:.3f}"
        )
    )
    verified_rows.append(
        {
            "Tác tử": label,
            "Checkpoint": path.name,
            "Board": meta.board_config.name,
            "Type": meta.training_type,
            "Epochs / Episodes": meta.epochs_trained if meta.epochs_trained > 0 else meta.episodes_trained,
            "Split / Params": meta.split_policy or f"states={meta.reward_summary.get('num_states', meta.model_config.get('num_parameters'))}",
            "Key Metric": metric_note,
            "SHA-256 (12 chars)": meta.weights_sha256[:12] + "...",
            "Trạng thái": "VERIFIED OK",
        }
    )

print("=== Bảng kiểm định tính toàn vẹn Checkpoint & Sidecar *.meta.json ===")
display(pd.DataFrame(verified_rows))

# Kiểm thử cơ chế chống nạp nhầm bàn cờ (board mismatch guard)
try:
    verify_checkpoint(CHECKPOINTS_DIR / "strongpolicynet_4x4_k4.pt", board_config=board_3x3_cfg)
except CheckpointVerificationError as exc:
    print(f"[Guard Check] Từ chối nạp chéo kích thước bàn cờ đúng thiết kế: {exc}")

# Khởi tạo các tác tử Cấp 5 & Cấp 6 từ trọng số đã xác minh
fast_policy_agent = PolicyNetworkAgent(
    checkpoint_path=CHECKPOINTS_DIR / "fastpolicynet_4x4_k4.pt",
    board_config=board_4x4_cfg,
    sample=False,
    device="cpu",
    name="L5_FastPolicy",
    seed=42,
)
strong_policy_agent = PolicyNetworkAgent(
    checkpoint_path=CHECKPOINTS_DIR / "strongpolicynet_4x4_k4.pt",
    board_config=board_4x4_cfg,
    sample=False,
    device="cpu",
    name="L5_StrongPolicy",
    seed=42,
)
value_greedy_agent = ValueGreedyAgent(
    checkpoint_path=CHECKPOINTS_DIR / "valuenet_4x4_k4.pt",
    board_config=board_4x4_cfg,
    device="cpu",
    name="L5_ValueGreedy",
    seed=42,
)
dqn_agent = DQNAgent(
    checkpoint_path=CHECKPOINTS_DIR / "dqnnet_4x4_k4_seed42.pt",
    board_config=board_4x4_cfg,
    epsilon=0.0,
    device="cpu",
    name="L6_DQN_seed42",
    seed=42,
)
tabular_q_agent = TabularQAgent(
    checkpoint_path=CHECKPOINTS_DIR / "tabular_q_3x3_k3_seed42.pkl",
    board_config=board_3x3_cfg,
    epsilon=0.0,
    name="L6_TabularQ_seed42",
    seed=42,
)

# Kiểm chứng Action Masking trên thế cờ mẫu (các ô đã đánh không được xuất hiện trong lựa chọn)
mask_test_4x4 = ExtendedTicTacToe(4, 4, 4)
for mv in [5, 6, 9, 10]:
    mask_test_4x4.step(mv)
legal_4x4 = set(mask_test_4x4.legal_actions())

mask_rows: list[dict[str, Any]] = []
for ag in (fast_policy_agent, strong_policy_agent, value_greedy_agent, dqn_agent):
    dec = ag.select_action(mask_test_4x4)
    assert dec.action in legal_4x4, f"Action {dec.action} không hợp lệ!"
    assert set(dec.action_scores.keys()) == legal_4x4, "Action masking chưa loại hết ô đã chiếm!"
    r, c = action_to_coords(dec.action, 4)
    mask_rows.append(
        {
            "Agent": ag.name,
            "Board": "4x4_k4",
            "Nước chọn": f"({r},{c}) [#{dec.action}]",
            "Value/Score": round(float(dec.value_estimate or 0.0), 4),
            "Latency (ms)": round(dec.latency_ms, 3),
            "Action Masking": f"PASS ({len(dec.action_scores)}/{len(legal_4x4)} legal)",
        }
    )

mask_test_3x3 = ExtendedTicTacToe(3, 3, 3)
for mv in [4, 0]:
    mask_test_3x3.step(mv)
dec_tab = tabular_q_agent.select_action(mask_test_3x3)
assert dec_tab.action in set(mask_test_3x3.legal_actions())
r3, c3 = action_to_coords(dec_tab.action, 3)
mask_rows.append(
    {
        "Agent": tabular_q_agent.name,
        "Board": "3x3_k3",
        "Nước chọn": f"({r3},{c3}) [#{dec_tab.action}]",
        "Value/Score": round(float(dec_tab.value_estimate or 0.0), 4),
        "Latency (ms)": round(dec_tab.latency_ms, 3),
        "Action Masking": f"PASS ({len(dec_tab.action_scores)}/{len(mask_test_3x3.legal_actions())} legal)",
    }
)
display(pd.DataFrame(mask_rows))

=== Bảng kiểm định tính toàn vẹn Checkpoint & Sidecar *.meta.json ===


,Tác tử,Checkpoint,Board,Type,Epochs / Episodes,Split / Params,Key Metric,SHA-256 (12 chars),Trạng thái
0,L5 FastPolicyNet,fastpolicynet_4x4_k4.pt,4x4_k4,supervised,40,game_id_70_15_15,val_acc=0.8008,9e0e0c4f51b3...,VERIFIED OK
1,L5 StrongPolicyNet,strongpolicynet_4x4_k4.pt,4x4_k4,supervised,40,game_id_70_15_15,val_acc=0.8021,b7737ef6891e...,VERIFIED OK
2,L5 ValueNet,valuenet_4x4_k4.pt,4x4_k4,supervised,40,game_id_70_15_15,val_loss=0.0534,1375837ecb8b...,VERIFIED OK
3,L6 DQNAgent (seed=42),dqnnet_4x4_k4_seed42.pt,4x4_k4,rl_dqn,20000,states=137616,eval_vs_rand=0.975,0db353609f65...,VERIFIED OK
4,L6 TabularQAgent (seed=42),tabular_q_3x3_k3_seed42.pkl,3x3_k3,rl_tabular,60000,states=4520,eval_vs_rand=0.925,95323af90e0e...,VERIFIED OK


[Guard Check] Từ chối nạp chéo kích thước bàn cờ đúng thiết kế: Checkpoint board_config mismatch: expected 3x3_k3 ({'rows': 3, 'cols': 3, 'win_k': 3}), found 4x4_k4 ({'rows': 4, 'cols': 4, 'win_k': 4}).


,Agent,Board,Nước chọn,Value/Score,Latency (ms),Action Masking
0,L5_FastPolicy,4x4_k4,"(0,1) [#1]",0.5652,1.053,PASS (12/12 legal)
1,L5_StrongPolicy,4x4_k4,"(0,1) [#1]",0.6956,0.479,PASS (12/12 legal)
2,L5_ValueGreedy,4x4_k4,"(0,2) [#2]",0.0462,1.166,PASS (12/12 legal)
3,L6_DQN_seed42,4x4_k4,"(3,1) [#13]",0.0444,0.342,PASS (12/12 legal)
4,L6_TabularQ_seed42,3x3_k3,"(0,1) [#1]",0.0000,0.051,PASS (7/7 legal)


In [9]:
# Trình diễn Cấp 7 (HybridMCTSAgent: PUCT + Policy Prior + LRU Cache) & Chứng minh hiệu chỉnh RQ1
from ttt_ai.agents import HybridMCTSAgent

hybrid_rollout = HybridMCTSAgent(
    policy_checkpoint_path=CHECKPOINTS_DIR / "strongpolicynet_4x4_k4.pt",
    board_config=board_4x4_cfg,
    n_simulations=200,
    c_puct=1.5,
    leaf_eval_mode="rollout",
    max_cache_size=4096,
    device="cpu",
    name="L7_HybridMCTS_Rollout",
    seed=42,
)
hybrid_mixed = HybridMCTSAgent(
    policy_checkpoint_path=CHECKPOINTS_DIR / "strongpolicynet_4x4_k4.pt",
    value_checkpoint_path=CHECKPOINTS_DIR / "valuenet_4x4_k4.pt",
    board_config=board_4x4_cfg,
    n_simulations=200,
    c_puct=1.5,
    leaf_eval_mode="mixed",
    value_weight=0.5,
    max_cache_size=4096,
    device="cpu",
    name="L7_HybridMCTS_Mixed",
    seed=42,
)

# 1. Thế cờ trung cuộc thực chiến trích từ Case 2 (ply 9, `L6_DQN__vs__L7_HybridMCTS__p0000__g0`)
# Lượt Player O (7 ô trống): đánh vào (0,2) [#2] tạo ngay thế đôi (hàng 0 + cột 2) dẫn tới chiến thắng!
midgame_prefix = [5, 6, 9, 10, 13, 1, 4, 0, 15]
midgame_game = ExtendedTicTacToe(4, 4, 4)
for mv in midgame_prefix:
    midgame_game.step(mv)

l4_mcts.reset(seed=42)
hybrid_rollout.clear_cache()
hybrid_rollout.reset(seed=42)
hybrid_mixed.clear_cache()
hybrid_mixed.reset(seed=42)

dec_policy_mid = strong_policy_agent.select_action(midgame_game)
dec_pure_mid = l4_mcts.select_action(midgame_game)
dec_rollout_mid = hybrid_rollout.select_action(midgame_game)
dec_hybrid_mid = hybrid_mixed.select_action(midgame_game)

print("=== THẾ CỜ 1 — CASE 2 (ply 9, lượt O — nước đôi quyết định tại (0,2) [#2]) ===")
print(midgame_game.render())
print(
    f"\n- L5 StrongPolicy (0 sims) chọn: {action_to_coords(dec_policy_mid.action, 4)} [#{dec_policy_mid.action}] "
    f"(P={dec_policy_mid.action_scores[dec_policy_mid.action]:.3f}; P(#2)={dec_hybrid_mid.prior_probs[2]:.3f})"
)
print(
    f"- L4 Pure MCTS (200 sims) chọn:   {action_to_coords(dec_pure_mid.action, 4)} [#{dec_pure_mid.action}] "
    f"(Visit Share={dec_pure_mid.action_scores[dec_pure_mid.action]:.3f})"
)
print(
    f"- L7 Hybrid Rollout (200 sims):   {action_to_coords(dec_rollout_mid.action, 4)} [#{dec_rollout_mid.action}] "
    f"(Visit Share={dec_rollout_mid.action_scores[dec_rollout_mid.action]:.3f})"
)
print(
    f"- L7 Hybrid Mixed (200 sims):     {action_to_coords(dec_hybrid_mid.action, 4)} [#{dec_hybrid_mid.action}] "
    f"(Visit Share={dec_hybrid_mid.action_scores[dec_hybrid_mid.action]:.3f}, "
    f"Prior={dec_hybrid_mid.prior_probs[dec_hybrid_mid.action]:.3f}, Q_root={dec_hybrid_mid.value_estimate:+.4f})"
)

df_mid_compare = pd.DataFrame(
    [
        {
            "Hạng PUCT": rank,
            "Ô (r, c) [#idx]": f"({action_to_coords(act, 4)[0]},{action_to_coords(act, 4)[1]}) [#{act}]",
            "prior_probs P(s,a)": round(dec_hybrid_mid.prior_probs[act], 4),
            "action_scores (Hybrid Mixed)": round(visit_share, 4),
            "Hybrid Rollout Visit": round(dec_rollout_mid.action_scores[act], 4),
            "Pure MCTS Visit": round(dec_pure_mid.action_scores[act], 4),
            "Hiệu chỉnh (Visit - Prior)": f"{visit_share - dec_hybrid_mid.prior_probs[act]:+.4f}",
        }
        for rank, (act, visit_share) in enumerate(dec_hybrid_mid.top_actions(k=5), start=1)
    ]
)
display(df_mid_compare)

# 2. Thế cờ chiến thuật thực chiến `L4_MCTS__vs__L7_HybridMCTS__p0041__g1` (ply 10, lượt X)
# StrongPolicyNet tự tin tới 97.4% vào #14 (P(#13)=0.021), bỏ lỡ nước thắng ngay của X tại cột 1: (3,1) [#13]!
# HybridMCTS (PUCT) đảo ngược hoàn toàn Prior cực đoan này để dồn 76.5% lượt thăm vào #13.
tactical_prefix = [0, 10, 3, 11, 5, 8, 9, 6, 1, 2]
tactical_game = ExtendedTicTacToe(4, 4, 4)
for mv in tactical_prefix:
    tactical_game.step(mv)

l4_mcts.reset(seed=42)
hybrid_rollout.reset(seed=42)
hybrid_mixed.reset(seed=42)

dec_policy_tac = strong_policy_agent.select_action(tactical_game)
dec_pure_tac = l4_mcts.select_action(tactical_game)
dec_rollout_tac = hybrid_rollout.select_action(tactical_game)
dec_hybrid_tac = hybrid_mixed.select_action(tactical_game)

# Minh họa tái sử dụng nút cây qua LRU Cache khi đi tiếp trong Case 2 (ply 11)
case2_ply11 = midgame_game.clone()
case2_ply11.step(2)
case2_ply11.step(14)
_ = hybrid_mixed.select_action(case2_ply11)

print("\n=== THẾ CỜ 2 — `L4_MCTS vs L7_HybridMCTS p0041__g1` (ply 10, lượt X — đảo ngược Prior 97.4% sai lầm) ===")
print(tactical_game.render())
print(
    f"\n- L5 StrongPolicy (0 sims) chọn: {action_to_coords(dec_policy_tac.action, 4)} [#{dec_policy_tac.action}] "
    f"(P={dec_policy_tac.action_scores[dec_policy_tac.action]:.3f}; P(#13)={dec_hybrid_tac.prior_probs[13]:.3f})"
)
print(
    f"- L7 Hybrid Mixed (200 sims):     {action_to_coords(dec_hybrid_tac.action, 4)} [#{dec_hybrid_tac.action}] "
    f"(Visit Share={dec_hybrid_tac.action_scores[dec_hybrid_tac.action]:.3f}, "
    f"Prior={dec_hybrid_tac.prior_probs[dec_hybrid_tac.action]:.3f}, Q_root={dec_hybrid_tac.value_estimate:+.4f})"
)
print(
    f"Telemetry: latency={dec_hybrid_tac.latency_ms:.2f} ms | sims={dec_hybrid_tac.simulations} | "
    f"nodes={dec_hybrid_tac.nodes_visited} | depth={dec_hybrid_tac.depth_reached}"
)
print(
    f"LRU Cache Stats (tổng hợp sau các truy vấn): hits={hybrid_mixed.cache_hits}, misses={hybrid_mixed.cache_misses}, "
    f"hit_rate={hybrid_mixed.cache_hit_rate * 100:.1f}%, cache_size={hybrid_mixed.cache_size}/{hybrid_mixed.max_cache_size}"
)

tac_rows = []
for act in sorted(tactical_game.legal_actions(), key=lambda a: -dec_hybrid_tac.action_scores[a])[:5]:
    r, c = action_to_coords(act, 4)
    tac_rows.append(
        {
            "Ô (r, c) [#idx]": f"({r},{c}) [#{act}]",
            "prior_probs P(s,a)": round(dec_hybrid_tac.prior_probs[act], 4),
            "action_scores (Hybrid Mixed)": round(dec_hybrid_tac.action_scores[act], 4),
            "Hybrid Rollout Visit": round(dec_rollout_tac.action_scores[act], 4),
            "Pure MCTS Visit": round(dec_pure_tac.action_scores[act], 4),
            "Hiệu chỉnh (Visit - Prior)": f"{dec_hybrid_tac.action_scores[act] - dec_hybrid_tac.prior_probs[act]:+.4f}",
        }
    )
display(pd.DataFrame(tac_rows))

=== THẾ CỜ 1 — CASE 2 (ply 9, lượt O — nước đôi quyết định tại (0,2) [#2]) ===
Board: 4x4 (K=4) | Turn: Player O (move #9)
     0   1   2   3 
   +---+---+---+---+
 0 | O | O | . | . |
   +---+---+---+---+
 1 | X | X | O | . |
   +---+---+---+---+
 2 | . | X | O | . |
   +---+---+---+---+
 3 | . | X | . | X |
   +---+---+---+---+

- L5 StrongPolicy (0 sims) chọn: (0, 3) [#3] (P=0.326; P(#2)=0.209)
- L4 Pure MCTS (200 sims) chọn:   (0, 2) [#2] (Visit Share=0.245)
- L7 Hybrid Rollout (200 sims):   (0, 2) [#2] (Visit Share=0.920)
- L7 Hybrid Mixed (200 sims):     (0, 2) [#2] (Visit Share=0.890, Prior=0.209, Q_root=+0.9568)


,Hạng PUCT,"Ô (r, c) [#idx]","prior_probs P(s,a)",action_scores (Hybrid Mixed),Hybrid Rollout Visit,Pure MCTS Visit,Hiệu chỉnh (Visit - Prior)
0,1,"(0,2) [#2]",0.2087,0.89,0.920,0.245,+0.6813
1,2,"(3,2) [#14]",0.3091,0.05,0.035,0.210,-0.2591
2,3,"(0,3) [#3]",0.3260,0.03,0.025,0.100,-0.2960
3,4,"(3,0) [#12]",0.1437,0.03,0.020,0.185,-0.1137
4,5,"(1,3) [#7]",0.0122,0.00,0.000,0.110,-0.0122



=== THẾ CỜ 2 — `L4_MCTS vs L7_HybridMCTS p0041__g1` (ply 10, lượt X — đảo ngược Prior 97.4% sai lầm) ===
Board: 4x4 (K=4) | Turn: Player X (move #10)
     0   1   2   3 
   +---+---+---+---+
 0 | X | X | O | X |
   +---+---+---+---+
 1 | . | X | O | . |
   +---+---+---+---+
 2 | O | X | O | O |
   +---+---+---+---+
 3 | . | . | . | . |
   +---+---+---+---+

- L5 StrongPolicy (0 sims) chọn: (3, 2) [#14] (P=0.974; P(#13)=0.021)
- L7 Hybrid Mixed (200 sims):     (3, 1) [#13] (Visit Share=0.765, Prior=0.021, Q_root=+1.0000)
Telemetry: latency=6.22 ms | sims=200 | nodes=8 | depth=6
LRU Cache Stats (tổng hợp sau các truy vấn): hits=1, misses=33, hit_rate=2.9%, cache_size=33/4096


,"Ô (r, c) [#idx]","prior_probs P(s,a)",action_scores (Hybrid Mixed),Hybrid Rollout Visit,Pure MCTS Visit,Hiệu chỉnh (Visit - Prior)
0,"(3,1) [#13]",0.0206,0.765,0.765,0.855,+0.7444
1,"(3,2) [#14]",0.9739,0.235,0.235,0.070,-0.7389
2,"(1,0) [#4]",0.0042,0.000,0.000,0.015,-0.0042
3,"(1,3) [#7]",0.0002,0.000,0.000,0.015,-0.0002
4,"(3,0) [#12]",0.0011,0.000,0.000,0.015,-0.0011


---
# PHẦN 4: ACTION HEATMAP & TRỰC QUAN HÓA THỜI GIAN THỰC

## Heatmap Bàn cờ Matplotlib 5 Mức màu & So sánh Policy Net vs MCTS vs Hybrid MCTS

Sử dụng hàm phân tầng `classify_heatmap_tier()` từ `ttt_ai.ui.theme` để ánh xạ `action_scores` thành **5 mức màu nhất quán với Rich CLI Dashboard**:
* **Tier 0 (Occupied):** Ô đã có quân `X` (xanh lam đậm) hoặc `O` (đỏ cam đậm) trên nền xám.
* **Tier 1 (Low):** Nước đi ưu tiên thấp (`normalized < 0.25`).
* **Tier 2 (Medium):** Nước đi trung bình (`0.25 <= normalized < 0.60`).
* **Tier 3 (High):** Nước đi mạnh (`normalized >= 0.60`).
* **Tier 4 (Top-1):** Nước đi được tác tử lựa chọn (vàng sáng nổi bật kèm viền đỏ/sao).

In [10]:
# Hàm vẽ Action Heatmap 5 mức màu & So sánh trực quan trên thế cờ trung cuộc + Case 1
from ttt_ai.ui.theme import classify_heatmap_tier

HEATMAP_TIER_COLORS: dict[int, str] = {
    0: "#E5E7EB",  # Tier 0: Occupied
    1: "#F8FAFC",  # Tier 1: Low
    2: "#FEF9C3",  # Tier 2: Medium
    3: "#86EFAC",  # Tier 3: High
    4: "#FACC15",  # Tier 4: Top-1 Selected
}


def plot_board_action_heatmap(
    game: ExtendedTicTacToe,
    action_scores: dict[int, float],
    chosen_action: int,
    title: str,
    *,
    ax: plt.Axes | None = None,
    value_fmt: str = "{:.3f}",
) -> plt.Axes:
    """Vẽ Heatmap bàn cờ M×N theo 5 mức màu từ `action_scores` và `classify_heatmap_tier()`."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.4, 4.4))

    rows, cols = game.rows, game.cols
    board = game.board
    legal = game.legal_actions()
    legal_vals = [float(action_scores.get(a, 0.0)) for a in legal]
    min_s = min(legal_vals) if legal_vals else 0.0
    max_s = max(legal_vals) if legal_vals else 0.0

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_aspect("equal")
    ax.invert_yaxis()

    for r in range(rows):
        for c in range(cols):
            act = coords_to_action(r, c, cols)
            cell_val = int(board[r, c])

            if cell_val != EMPTY:
                tier = 0
                face = HEATMAP_TIER_COLORS[tier]
                edge = "#9CA3AF"
                lw = 1.0
                mark = "X" if cell_val == PLAYER_X else "O"
                mark_color = PALETTE["blue"] if cell_val == PLAYER_X else PALETTE["vermilion"]
                ax.add_patch(mpatches.Rectangle((c, r), 1, 1, facecolor=face, edgecolor=edge, linewidth=lw))
                ax.text(c + 0.5, r + 0.46, mark, ha="center", va="center", fontsize=17, fontweight="bold", color=mark_color)
                ax.text(c + 0.5, r + 0.82, f"#{act}", ha="center", va="center", fontsize=7.5, color="#6B7280")
            else:
                score = float(action_scores.get(act, 0.0))
                is_top1 = act == chosen_action
                tier = classify_heatmap_tier(score, min_s, max_s, is_top1)
                face = HEATMAP_TIER_COLORS[tier]
                edge = "#B91C1C" if is_top1 else "#64748B"
                lw = 2.6 if is_top1 else 1.0
                ax.add_patch(mpatches.Rectangle((c, r), 1, 1, facecolor=face, edgecolor=edge, linewidth=lw, zorder=2 if is_top1 else 1))
                top_tag = " *" if is_top1 else ""
                ax.text(
                    c + 0.5,
                    r + 0.30,
                    f"({r},{c})#{act}{top_tag}",
                    ha="center",
                    va="center",
                    fontsize=7.6,
                    fontweight="bold" if is_top1 else "normal",
                    color="#111827",
                    zorder=3,
                )
                ax.text(
                    c + 0.5,
                    r + 0.66,
                    value_fmt.format(score),
                    ha="center",
                    va="center",
                    fontsize=10.5 if is_top1 else 9.5,
                    fontweight="bold" if tier >= 3 else "normal",
                    color="#991B1B" if is_top1 else "#1E293B",
                    zorder=3,
                )

    ax.set_xticks([c + 0.5 for c in range(cols)])
    ax.set_xticklabels([f"c={c}" for c in range(cols)])
    ax.set_yticks([r + 0.5 for r in range(rows)])
    ax.set_yticklabels([f"r={r}" for r in range(rows)])
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, pad=8, fontsize=10.2)
    return ax


# So sánh trực quan 3 phương pháp trên Thế cờ trung cuộc Case 2 (Hàng 1) & Thế cờ p0041__g1 (Hàng 2)
fig, axes = plt.subplots(2, 3, figsize=(14.5, 9.6), constrained_layout=True)

plot_board_action_heatmap(
    midgame_game,
    dec_policy_mid.action_scores,
    dec_policy_mid.action,
    f"Case 2 (ply 9): L5 StrongPolicyNet\n(Prior P -> #{dec_policy_mid.action}={dec_policy_mid.action_scores[dec_policy_mid.action]:.3f})",
    ax=axes[0, 0],
)
plot_board_action_heatmap(
    midgame_game,
    dec_pure_mid.action_scores,
    dec_pure_mid.action,
    f"Case 2 (ply 9): L4 Pure MCTS (UCT)\n(Visit -> #{dec_pure_mid.action}={dec_pure_mid.action_scores[dec_pure_mid.action]:.3f})",
    ax=axes[0, 1],
)
plot_board_action_heatmap(
    midgame_game,
    dec_hybrid_mid.action_scores,
    dec_hybrid_mid.action,
    f"Case 2 (ply 9): L7 Hybrid MCTS (PUCT)\n(Visit -> #{dec_hybrid_mid.action}={dec_hybrid_mid.action_scores[dec_hybrid_mid.action]:.3f}, Prior={dec_hybrid_mid.prior_probs[dec_hybrid_mid.action]:.3f})",
    ax=axes[0, 2],
)

plot_board_action_heatmap(
    tactical_game,
    dec_policy_tac.action_scores,
    dec_policy_tac.action,
    f"p0041__g1 (ply 10): L5 PolicyNet\n(Prior -> #{dec_policy_tac.action}={dec_policy_tac.action_scores[dec_policy_tac.action]:.3f}, bỏ lỡ thắng #13)",
    ax=axes[1, 0],
)
plot_board_action_heatmap(
    tactical_game,
    dec_pure_tac.action_scores,
    dec_pure_tac.action,
    f"p0041__g1 (ply 10): L4 Pure MCTS\n(Visit -> #{dec_pure_tac.action}={dec_pure_tac.action_scores[dec_pure_tac.action]:.3f})",
    ax=axes[1, 1],
)
plot_board_action_heatmap(
    tactical_game,
    dec_hybrid_tac.action_scores,
    dec_hybrid_tac.action,
    f"p0041__g1 (ply 10): L7 Hybrid MCTS (Mixed)\n(Sửa Prior 0.021 -> Visit #{dec_hybrid_tac.action}={dec_hybrid_tac.action_scores[dec_hybrid_tac.action]:.3f})",
    ax=axes[1, 2],
)

legend_patches = [
    mpatches.Patch(facecolor=HEATMAP_TIER_COLORS[4], edgecolor="#B91C1C", linewidth=1.8, label="Tier 4: Top-1 Selected"),
    mpatches.Patch(facecolor=HEATMAP_TIER_COLORS[3], edgecolor="#64748B", label="Tier 3: High (>=60% span)"),
    mpatches.Patch(facecolor=HEATMAP_TIER_COLORS[2], edgecolor="#64748B", label="Tier 2: Medium (25%–60%)"),
    mpatches.Patch(facecolor=HEATMAP_TIER_COLORS[1], edgecolor="#64748B", label="Tier 1: Low (<25% span)"),
    mpatches.Patch(facecolor=HEATMAP_TIER_COLORS[0], edgecolor="#9CA3AF", label="Tier 0: Occupied (X / O)"),
]
fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=5,
    frameon=True,
    bbox_to_anchor=(0.5, -0.025),
    fontsize=8.5,
)
fig.suptitle(
    "Action Heatmap 5 Mức màu: So sánh Phân phối Policy Prior vs MCTS UCT vs Hybrid PUCT",
    fontsize=12.8,
    fontweight="bold",
)
plt.show()

C:\Users\Ron\AppData\Local\Temp\ipykernel_28484\1132974874.py:163: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
# PHẦN 5: TÁI TẠO KẾT QUẢ THỰC NGHIỆM ĐỒ ÁN (`FIG 1–5` & `RQ1, RQ2, RQ3`)

## Nạp Raw JSONL Logs & Bảng Tổng hợp Thực nghiệm (*Zero Fabrication*)

Mọi bảng số liệu và biểu đồ `Fig 1..5` dưới đây được tính toán và tái hiện trực tiếp từ các tệp nhật ký thô (`logs/*.jsonl`) và bảng tổng hợp `reports/summary_metrics.csv`:
* **4 giải đấu Paired Round-Robin chính thức:** `16,800` ván (`4,200` ván/giải, cân bằng `50% X / 50% O`, `200` ván/cặp).
* **4 thực nghiệm Ablation (`AB Pruning`, `RQ1`, `RQ2`, `RQ3`):** `8,756` ván đối kháng có kiểm soát cùng learning curves đa seed (`logs/train_*.jsonl`).

In [11]:
# Đọc trực tiếp logs/tournament_*.jsonl, logs/ablation_*.jsonl và reports/summary_metrics.csv
# Lưu và khôi phục matplotlib backend vì `ttt_ai.benchmark` import `visualizer` (mặc định đặt `Agg` cho CLI export)
_prev_mpl_backend = matplotlib.get_backend()
from ttt_ai.benchmark.ablations import load_ablation_result
from ttt_ai.benchmark.metrics import summarize_tournament_log

if matplotlib.get_backend() != _prev_mpl_backend:
    try:
        matplotlib.use(_prev_mpl_backend, force=True)
    except Exception:
        pass
plt.rcParams.update(PLOT_RC)

summary_4x4 = summarize_tournament_log(
    LOGS_DIR / "tournament_4x4_k4_default.jsonl",
    n_bootstrap=1000,
    seed=42,
)
summary_3x3 = summarize_tournament_log(
    LOGS_DIR / "tournament_3x3_k3_default.jsonl",
    n_bootstrap=1000,
    seed=42,
)
ab1_res = load_ablation_result(LOGS_DIR / "ablation_ab_pruning.jsonl")
rq1_res = load_ablation_result(LOGS_DIR / "ablation_rq1_hybrid_mcts.jsonl")
rq2_res = load_ablation_result(LOGS_DIR / "ablation_rq2_seat_advantage.jsonl")
rq3_res = load_ablation_result(LOGS_DIR / "ablation_rq3_rl_generalization.jsonl")

df_metrics = pd.read_csv(REPORTS_DIR / "summary_metrics.csv")

log_inventory = pd.DataFrame(
    [
        {"Nguồn Log": "tournament_4x4_k4_default.jsonl", "Loại": "Tournament (4x4 K=4, default)", "Số trận": summary_4x4.total_matches, "Tác tử / Nhóm": len(summary_4x4.participants)},
        {"Nguồn Log": "tournament_3x3_k3_default.jsonl", "Loại": "Tournament (3x3 K=3, default)", "Số trận": summary_3x3.total_matches, "Tác tử / Nhóm": len(summary_3x3.participants)},
        {"Nguồn Log": "ablation_ab_pruning.jsonl", "Loại": "Ablation 1 (Alpha-Beta & Ordering)", "Số trận": ab1_res.total_matches, "Tác tử / Nhóm": len(ab1_res.summary.get("probe_by_depth", {}))},
        {"Nguồn Log": "ablation_rq1_hybrid_mcts.jsonl", "Loại": "Ablation 2 (RQ1 Hybrid MCTS)", "Số trận": rq1_res.total_matches, "Tác tử / Nhóm": len(rq1_res.summary.get("candidate_metrics", {}))},
        {"Nguồn Log": "ablation_rq2_seat_advantage.jsonl", "Loại": "Ablation 3 (RQ2 Seat Advantage)", "Số trận": rq2_res.total_matches, "Tác tử / Nhóm": len(rq2_res.summary.get("boards", {}))},
        {"Nguồn Log": "ablation_rq3_rl_generalization.jsonl", "Loại": "Ablation 4 (RQ3 RL vs Supervised)", "Số trận": rq3_res.total_matches, "Tác tử / Nhóm": len(rq3_res.summary.get("boards", {}))},
    ]
)
print(f"Đã đọc `reports/summary_metrics.csv` ({len(df_metrics)} dòng) & toàn bộ raw JSONL logs.")
display(log_inventory)

print("\n=== Bảng xếp hạng Giải đấu Chính `tournament_4x4_k4_default` (4,200 ván) ===")
df_4x4_lb = df_metrics[df_metrics["experiment_id"] == "tournament_4x4_k4_default"][
    ["agent_or_candidate", "wins", "draws", "losses", "score", "score_ci_lower", "score_ci_upper", "score_x", "score_o", "delta_score", "mean_latency_ms", "p95_latency_ms"]
].reset_index(drop=True)
display(df_4x4_lb)

Đã đọc `reports/summary_metrics.csv` (83 dòng) & toàn bộ raw JSONL logs.


,Nguồn Log,Loại,Số trận,Tác tử / Nhóm
0,tournament_4x4_k4_default.jsonl,"Tournament (4x4 K=4, default)",4200,7
1,tournament_3x3_k3_default.jsonl,"Tournament (3x3 K=3, default)",4200,7
2,ablation_ab_pruning.jsonl,Ablation 1 (Alpha-Beta & Ordering),316,3
3,ablation_rq1_hybrid_mcts.jsonl,Ablation 2 (RQ1 Hybrid MCTS),2760,18
4,ablation_rq2_seat_advantage.jsonl,Ablation 3 (RQ2 Seat Advantage),2080,3
5,ablation_rq3_rl_generalization.jsonl,Ablation 4 (RQ3 RL vs Supervised),3600,2



=== Bảng xếp hạng Giải đấu Chính `tournament_4x4_k4_default` (4,200 ván) ===


,agent_or_candidate,wins,draws,losses,score,score_ci_lower,score_ci_upper,score_x,score_o,delta_score,mean_latency_ms,p95_latency_ms
0,L4_MCTS,282,911,7,0.6146,0.6029,0.6275,0.6158,0.6133,0.0025,21.5401,61.8252
1,L2_RuleBased,203,997,0,0.5846,0.5737,0.5946,0.5925,0.5767,0.0158,0.1298,0.2185
2,L6_DQN,179,995,26,0.5637,0.5525,0.5746,0.5533,0.5742,-0.0208,0.3994,0.8519
3,L3_AlphaBeta,148,1052,0,0.5617,0.5529,0.5713,0.5717,0.5517,0.0200,12.7472,46.4771
4,L7_HybridMCTS,181,980,39,0.5592,0.5479,0.5713,0.5567,0.5617,-0.0050,47.7888,214.2065
5,L5_StrongPolicy,131,983,86,0.5188,0.5075,0.5304,0.5325,0.5050,0.0275,0.7295,1.7157
6,L1_Random,8,218,974,0.0975,0.0862,0.1096,0.1292,0.0658,0.0633,0.0553,0.0778


In [12]:
# Tái hiện Fig 1 (Ma trận đối đầu kép X vs O) & Fig 2 (Độ trễ suy luận thang log & Chi phí tìm kiếm)
_SHORT_LABELS: dict[str, str] = {
    "L1_Random": "L1 Random",
    "L2_RuleBased": "L2 RuleBased",
    "L3_AlphaBeta": "L3 AlphaBeta",
    "L4_MCTS": "L4 MCTS",
    "L5_StrongPolicy": "L5 Policy",
    "L6_DQN": "L6 DQN",
    "L6_TabularQ": "L6 TabularQ",
    "L7_HybridMCTS": "L7 HybridMCTS",
}


def _agent_order_key(agent_id: str) -> tuple[int, str]:
    digits = "".join(ch for ch in agent_id.split("_")[0] if ch.isdigit())
    return (int(digits) if digits else 99, agent_id)


def _ci_errors(vals: list[float], lows: list[float], highs: list[float]) -> tuple[list[float], list[float]]:
    return (
        [max(0.0, float(v) - float(lo)) for v, lo in zip(vals, lows)],
        [max(0.0, float(hi) - float(v)) for v, hi in zip(vals, highs)],
    )


# ---- FIG 1: Ma trận đối đầu kép As X vs As O (4x4 K=4, default) ----
agents_sorted = sorted(summary_4x4.participants, key=_agent_order_key)
n_agents = len(agents_sorted)
mat_x = np.full((n_agents, n_agents), np.nan, dtype=np.float64)
mat_o = np.full((n_agents, n_agents), np.nan, dtype=np.float64)
txt_x = [["-" for _ in range(n_agents)] for _ in range(n_agents)]
txt_o = [["-" for _ in range(n_agents)] for _ in range(n_agents)]

for i, a in enumerate(agents_sorted):
    for j, b in enumerate(agents_sorted):
        if i == j:
            continue
        h2h = summary_4x4.get_head_to_head(a, b)
        if h2h is None:
            continue
        if h2h.games_a_as_x > 0:
            sx = (h2h.wins_a_as_x + 0.5 * h2h.draws_a_as_x) / float(h2h.games_a_as_x)
            mat_x[i, j] = sx
            txt_x[i][j] = f"{sx:.2f}\n({h2h.wins_a_as_x}-{h2h.draws_a_as_x}-{h2h.losses_a_as_x})"
        if h2h.games_a_as_o > 0:
            so = (h2h.wins_a_as_o + 0.5 * h2h.draws_a_as_o) / float(h2h.games_a_as_o)
            mat_o[i, j] = so
            txt_o[i][j] = f"{so:.2f}\n({h2h.wins_a_as_o}-{h2h.draws_a_as_o}-{h2h.losses_a_as_o})"

short_names = [_SHORT_LABELS.get(a, a) for a in agents_sorted]
fig1, axes1 = plt.subplots(1, 2, figsize=(14.0, 6.2), constrained_layout=True)
cmap = plt.get_cmap("RdYlGn").with_extremes(bad="#E8ECEF")

im1 = None
for ax, mat, labels, sub in (
    (axes1[0], mat_x, txt_x, "Row Agent cầm X (Đi trước) vs Column (O)"),
    (axes1[1], mat_o, txt_o, "Row Agent cầm O (Đi sau) vs Column (X)"),
):
    im1 = ax.imshow(mat, cmap=cmap, vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_title(sub, pad=10)
    ax.set_xticks(range(n_agents))
    ax.set_xticklabels(short_names, rotation=32, ha="right")
    ax.set_yticks(range(n_agents))
    ax.set_yticklabels(short_names)
    ax.set_xlabel("Opponent")
    ax.set_ylabel("Evaluated Agent")
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
    for r in range(n_agents):
        for c in range(n_agents):
            val = mat[r, c]
            tcol = "#555555" if np.isnan(val) else ("white" if (val <= 0.22 or val >= 0.80) else "#111111")
            ax.text(c, r, labels[r][c], ha="center", va="center", fontsize=7.5, fontweight="bold" if r != c else "normal", color=tcol)

if im1 is not None:
    cbar1 = fig1.colorbar(im1, ax=axes1, shrink=0.88, pad=0.02)
    cbar1.set_label("Match Score = (W + 0.5·D) / N")
fig1.suptitle(
    f"Fig 1: Dual Head-to-Head Score Matrix — 4x4 K=4 (default, N={summary_4x4.total_matches} matches)",
    fontsize=12.5,
    fontweight="bold",
)
plt.show()

# ---- FIG 2: Độ trễ suy luận (Mean & P95, thang log) & Chi phí tìm kiếm (Nodes / Simulations) ----
ordered_metrics = sorted(summary_4x4.agent_metrics.values(), key=lambda m: _agent_order_key(m.agent_id))
labels_2 = [_SHORT_LABELS.get(m.agent_id, m.agent_id) for m in ordered_metrics]
x2 = np.arange(len(ordered_metrics), dtype=np.float64)
means_2 = [max(m.mean_latency_ms, 1e-3) for m in ordered_metrics]
p95s_2 = [max(m.p95_latency_ms, 1e-3) for m in ordered_metrics]
nodes_2 = [m.avg_nodes_visited for m in ordered_metrics]
sims_2 = [m.avg_simulations for m in ordered_metrics]

fig2, (ax_lat, ax_cost) = plt.subplots(1, 2, figsize=(13.5, 5.4), constrained_layout=True)

bars_lat = ax_lat.bar(x2, means_2, width=0.52, color=PALETTE["blue"], alpha=0.88, edgecolor="#1B2631", lw=0.7, label="Mean Latency (ms)", zorder=3)
for xi, mu, p95 in zip(x2, means_2, p95s_2):
    ax_lat.plot([xi, xi], [mu, p95], color=PALETTE["vermilion"], lw=1.6, zorder=4)
ax_lat.scatter(x2, p95s_2, color=PALETTE["vermilion"], marker="D", s=46, zorder=5, label="P95 Latency (ms)")
ax_lat.axhline(10.0, color=PALETTE["amber"], ls="--", lw=1.1, label="10 ms Budget", zorder=2)
ax_lat.axhline(50.0, color=PALETTE["purple"], ls=":", lw=1.3, label="50 ms Budget", zorder=2)
ax_lat.set_yscale("log")
ax_lat.set_ylim(0.01, max(max(p95s_2), 60.0) * 2.8)
ax_lat.set_xticks(x2)
ax_lat.set_xticklabels(labels_2, rotation=25, ha="right")
ax_lat.set_ylabel("Move Latency (ms, log scale)")
ax_lat.set_title("(a) Per-Move Latency: Mean vs P95")
ax_lat.grid(axis="y", alpha=0.25, ls="--", which="both")
ax_lat.legend(loc="upper left", frameon=True, ncol=2)
for bar, m_obj, p95 in zip(bars_lat, ordered_metrics, p95s_2):
    ax_lat.text(bar.get_x() + bar.get_width() / 2.0, max(bar.get_height(), p95) * 1.18, f"{m_obj.mean_latency_ms:.2f}\n(p95:{m_obj.p95_latency_ms:.1f})", ha="center", va="bottom", fontsize=7.0)

w2 = 0.36
b_nodes = ax_cost.bar(x2 - w2 / 2.0, [max(v, 0.5) if v > 0 else 0.0 for v in nodes_2], width=w2, color=PALETTE["green"], alpha=0.90, edgecolor="#1B2631", lw=0.7, label="Avg Nodes Visited / Move", zorder=3)
b_sims = ax_cost.bar(x2 + w2 / 2.0, [max(v, 0.5) if v > 0 else 0.0 for v in sims_2], width=w2, color=PALETTE["amber"], alpha=0.90, edgecolor="#1B2631", lw=0.7, label="Avg Simulations / Move", zorder=3)
ax_cost.set_yscale("log")
ax_cost.set_ylim(0.5, max(max(nodes_2), max(sims_2), 10.0) * 3.0)
ax_cost.set_xticks(x2)
ax_cost.set_xticklabels(labels_2, rotation=25, ha="right")
ax_cost.set_ylabel("Search Operations / Move (log scale)")
ax_cost.set_title("(b) Average Search Cost: Nodes & Simulations")
ax_cost.grid(axis="y", alpha=0.25, ls="--", which="both")
ax_cost.legend(loc="upper left", frameon=True)
for bar, val in zip(b_nodes, nodes_2):
    if val > 0.0:
        ax_cost.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() * 1.15, f"{val:.0f}" if val >= 10 else f"{val:.1f}", ha="center", va="bottom", fontsize=7.3)
for bar, val in zip(b_sims, sims_2):
    if val > 0.0:
        ax_cost.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() * 1.15, f"{val:.0f}", ha="center", va="bottom", fontsize=7.3, color="#7E5109")

fig2.suptitle("Fig 2: Computational Cost & Inference Latency — 4x4 K=4 (default)", fontsize=12.5, fontweight="bold")
plt.show()

C:\Users\Ron\AppData\Local\Temp\ipykernel_28484\3202044886.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Ron\AppData\Local\Temp\ipykernel_28484\3202044886.py:132: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# Tái hiện Fig 3: RQ1 (Hybrid MCTS vs Pure MCTS & Policy-only) + Alpha-Beta Pruning Ablation
cand_metrics = rq1_res.summary["candidate_metrics"]
probe_by_depth = ab1_res.summary["probe_by_depth"]

fig3, (ax_sim, ax_time, ax_ab) = plt.subplots(1, 3, figsize=(16.0, 5.2), constrained_layout=True)

families = (
    ("PureMCTS", "Pure MCTS (UCT)", PALETTE["blue"], "o", "-"),
    ("HybridRollout", "Hybrid MCTS (Rollout)", PALETTE["amber"], "s", "--"),
    ("HybridMixed", "Hybrid MCTS (Mixed)", PALETTE["green"], "^", "-"),
)
for fam, label, col, marker, ls in families:
    pts = sorted(
        ((int(m["budget_val"]), m) for m in cand_metrics.values() if m.get("family") == fam and m.get("budget_type") == "simulations"),
        key=lambda item: item[0],
    )
    xs = [p[0] for p in pts]
    ys = [float(p[1]["score"]) for p in pts]
    lo = [float(p[1]["score_ci"][0]) for p in pts]
    hi = [float(p[1]["score_ci"][1]) for p in pts]
    ax_sim.plot(xs, ys, marker=marker, ls=ls, lw=2.0, color=col, label=label)
    ax_sim.fill_between(xs, lo, hi, color=col, alpha=0.14)

for cid, lbl, col, ls in (
    ("Policy_Strong", "Policy Strong (0 sims)", PALETTE["purple"], ":"),
    ("Policy_Fast", "Policy Fast (0 sims)", PALETTE["sky"], "-."),
    ("Value_Greedy", "Value Greedy (1-ply)", PALETTE["vermilion"], ":"),
):
    if cid in cand_metrics:
        val = float(cand_metrics[cid]["score"])
        ax_sim.axhline(val, color=col, ls=ls, lw=1.4, label=f"{lbl}: {val:.3f}")

ax_sim.set_title("(a) RQ1: Score vs Simulation Budget")
ax_sim.set_xlabel("MCTS Simulations per Move")
ax_sim.set_ylabel("Mean Score vs Opponent Pool (95% CI)")
ax_sim.set_ylim(0.20, 0.72)
ax_sim.grid(alpha=0.25, ls="--")
ax_sim.legend(loc="lower right", frameon=True, fontsize=7.5)

# Panel (b): Fixed Time Budgets (10 ms vs 50 ms)
time_budgets = sorted({float(m["budget_val"]) for m in cand_metrics.values() if m.get("budget_type") == "time_ms"})
tx = np.arange(len(time_budgets), dtype=np.float64)
bar_w = 0.24
for idx, (fam, label, col, _, _) in enumerate(families):
    vals, lows, highs, slbls = [], [], [], []
    for tb in time_budgets:
        matched = [m for m in cand_metrics.values() if m.get("family") == fam and m.get("budget_type") == "time_ms" and abs(float(m.get("budget_val", -1)) - tb) < 1e-6]
        m = matched[0]
        vals.append(float(m["score"]))
        lows.append(float(m["score_ci"][0]))
        highs.append(float(m["score_ci"][1]))
        slbls.append(f"{float(m.get('avg_simulations', 0)):.0f}s")
    bars = ax_time.bar(tx + (idx - 1) * bar_w, vals, width=bar_w, color=col, alpha=0.88, edgecolor="#1B2631", lw=0.7, yerr=_ci_errors(vals, lows, highs), capsize=3, label=label, zorder=3)
    for b, v, hi, sl in zip(bars, vals, highs, slbls):
        ax_time.text(b.get_x() + b.get_width() / 2.0, max(v, hi) + 0.008, f"{v:.3f}\n({sl})", ha="center", va="bottom", fontsize=6.8)

ax_time.set_xticks(tx)
ax_time.set_xticklabels([f"{int(tb)} ms / move" for tb in time_budgets])
ax_time.set_ylim(0.32, 0.74)
ax_time.set_ylabel("Mean Score vs Opponent Pool (95% CI)")
ax_time.set_title("(b) RQ1: Score under Fixed Time Budget")
ax_time.grid(axis="y", alpha=0.25, ls="--")
ax_time.legend(loc="lower right", frameon=True, fontsize=7.5)

# Panel (c): Alpha-Beta Pruning & Move Ordering
depth_keys = sorted(probe_by_depth.keys(), key=lambda k: int(probe_by_depth[k]["depth"]))
dx = np.arange(len(depth_keys), dtype=np.float64)
bw = 0.25
mm_vals = [float(probe_by_depth[k]["mm_avg_nodes"]) for k in depth_keys]
ab_vals = [float(probe_by_depth[k]["ab_no_order_avg_nodes"]) for k in depth_keys]
ord_vals = [float(probe_by_depth[k]["ab_order_avg_nodes"]) for k in depth_keys]

ax_ab.bar(dx - bw, mm_vals, width=bw, color=PALETTE["vermilion"], edgecolor="#1B2631", lw=0.7, label="Raw Minimax")
b_ab = ax_ab.bar(dx, ab_vals, width=bw, color=PALETTE["amber"], edgecolor="#1B2631", lw=0.7, label="Alpha-Beta (No Order)")
b_ord = ax_ab.bar(dx + bw, ord_vals, width=bw, color=PALETTE["green"], edgecolor="#1B2631", lw=0.7, label="Alpha-Beta + Move Ordering")
ax_ab.set_yscale("log")
ax_ab.set_ylim(10.0, max(mm_vals) * 3.5)
ax_ab.set_xticks(dx)
ax_ab.set_xticklabels([f"Depth {probe_by_depth[k]['depth']}" for k in depth_keys])
ax_ab.set_ylabel("Average Visited Nodes (log scale)")
ax_ab.set_title("(c) Alpha-Beta Pruning & Ordering (4x4 K=4)")
ax_ab.grid(axis="y", alpha=0.25, ls="--", which="both")
ax_ab.legend(loc="upper left", frameon=True, fontsize=7.5)

for bar, k in zip(b_ab, depth_keys):
    ax_ab.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() * 1.15, f"-{float(probe_by_depth[k]['ab_reduction_pct']):.0f}%", ha="center", va="bottom", fontsize=7.0, color="#7E5109")
for bar, k in zip(b_ord, depth_keys):
    ax_ab.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() * 1.15, f"-{float(probe_by_depth[k]['ab_order_reduction_pct']):.1f}%", ha="center", va="bottom", fontsize=7.2, fontweight="bold", color="#0B5345")

fig3.suptitle("Fig 3: Search Guidance (RQ1) & Alpha-Beta Pruning Efficiency (4x4 K=4)", fontsize=12.5, fontweight="bold")
plt.show()

C:\Users\Ron\AppData\Local\Temp\ipykernel_28484\161055953.py:91: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Thảo luận Chuyên sâu RQ1 — Hiệu quả Hybrid MCTS vs Pure MCTS & Policy-only

1. **Ưu thế vượt trội ở ngân sách tìm kiếm thấp (`50 simulations`):**
   * Tại `n_simulations = 50` trên `4×4, K=4`, **`HybridRollout_sim50`** đạt điểm **`0.583 [0.542, 0.625]`** (chỉ `5` ván thua/120 ván) và **`HybridMixed_sim50`** đạt **`0.575 [0.533, 0.621]`** (độ sâu cây đạt `6.46` plies).
   * Trong khi đó, **`PureMCTS_sim50`** chỉ đạt **`0.529 [0.471, 0.588]`** (`22` ván thua, độ sâu chỉ `2.42` plies), còn mạng đơn lẻ không tìm kiếm **`Policy_Strong`** đạt **`0.525`** (`Value_Greedy` chỉ `0.296`). Nhờ xác suất tiên nghiệm $P(s, a)$ dẫn nhánh theo PUCT, Hybrid MCTS tránh lãng phí mô phỏng vào các nước yếu ngay từ những bước đầu tiên.
2. **Đánh đổi khi tăng ngân sách hoặc siết thời gian CPU (`150–400 sims` & `10 ms / 50 ms`):**
   * Khi tăng lên `150–400 simulations`, `PureMCTS` (`0.604` → `0.621`) bắt kịp và nhỉnh hơn `HybridRollout` (`0.579` → `0.558`) / `HybridMixed` (`0.600`), vì với `4×4` nhánh nhỏ, rollout ngẫu nhiên nhanh cho phép `PureMCTS` chạy đủ rộng để tự sửa sai, trong khi forward `StrongPolicyNet` trên CPU tiêu tốn nhiều mili-giây hơn (`3.79 ms` vs `8.78 ms` ở `50 sims`; ở `10 ms/move`, `PureMCTS` chạy được `190.7 sims` so với `145.9 sims` của `HybridMixed`).
   * **Vai trò của LRU Prior Cache:** Bộ đệm LRU theo hash canonical giúp giảm `47.0%–69.0%` lượt suy luận mạng dư thừa, giữ cho `HybridMixed_time10ms` đạt `0.583` ngang `HybridMixed_time50ms`.
   * Ngoài ra trên đối chứng `3×3, K=3` (`default`), **`L7_HybridMCTS`** đứng **Hạng #1 toàn giải** với **`0.637 [0.624, 0.650]`** (`328` thắng, `872` hòa, `0` thua).
3. **Bổ trợ từ Ablation 1 (Alpha-Beta Pruning & Move Ordering):** Tại độ sâu `d=4`, bật Alpha-Beta giảm số nút duyệt từ `23,171.0` xuống `2,035.5` (**`-91.22%`**), và kết hợp *Move Ordering* kéo xuống còn `1,225.0` (**`-94.71%`**) trong khi giữ nguyên tuyệt đối giá trị nút gốc (`Exact Root Value = YES`).

In [14]:
# Tái hiện Fig 4: RQ2 First-Mover Advantage (4x4 K=3 vs 3x3 K=3 vs 4x4 K=4)
boards_rq2 = rq2_res.summary["boards"]
preferred_order = ("4x4_k3", "3x3_k3", "4x4_k4")
board_keys = [k for k in preferred_order if k in boards_rq2] + [k for k in sorted(boards_rq2.keys()) if k not in preferred_order]
board_titles = {"4x4_k3": "4x4, K=3", "3x3_k3": "3x3, K=3", "4x4_k4": "4x4, K=4"}
board_colors = {"4x4_k3": PALETTE["vermilion"], "3x3_k3": PALETTE["amber"], "4x4_k4": PALETTE["blue"]}

fig4, (ax_global, ax_levels) = plt.subplots(1, 2, figsize=(14.5, 5.5), gridspec_kw={"width_ratios": [1.0, 1.55]}, constrained_layout=True)

gx = np.arange(len(board_keys), dtype=np.float64)
x_rates = [100.0 * float(boards_rq2[k]["seat_advantage"]["x_win_rate"]) for k in board_keys]
d_rates = [100.0 * float(boards_rq2[k]["seat_advantage"]["draw_rate"]) for k in board_keys]
o_rates = [100.0 * float(boards_rq2[k]["seat_advantage"]["o_win_rate"]) for k in board_keys]

ax_global.bar(gx, x_rates, width=0.52, color=PALETTE["blue"], label="X Wins (1st Mover)", edgecolor="#1B2631", lw=0.7)
ax_global.bar(gx, d_rates, width=0.52, bottom=x_rates, color="#D5D8DC", label="Draws", edgecolor="#1B2631", lw=0.7)
ax_global.bar(gx, o_rates, width=0.52, bottom=[xr + dr for xr, dr in zip(x_rates, d_rates)], color=PALETTE["vermilion"], label="O Wins (2nd Mover)", edgecolor="#1B2631", lw=0.7)

for xi, k, xr, dr, orate in zip(gx, board_keys, x_rates, d_rates, o_rates):
    sa = boards_rq2[k]["seat_advantage"]
    ds, ci = float(sa["delta_score"]), sa["delta_score_ci"]
    if xr >= 9.0:
        ax_global.text(xi, xr / 2.0, f"{xr:.1f}%", ha="center", va="center", color="white", fontweight="bold", fontsize=8.0)
    if dr >= 9.0:
        ax_global.text(xi, xr + dr / 2.0, f"{dr:.1f}%", ha="center", va="center", color="#222222", fontweight="bold", fontsize=8.0)
    if orate >= 9.0:
        ax_global.text(xi, xr + dr + orate / 2.0, f"{orate:.1f}%", ha="center", va="center", color="white", fontweight="bold", fontsize=8.0)
    ax_global.text(xi, 101.8, f"ΔScore={ds:+.3f}\n[{ci[0]:+.3f}, {ci[1]:+.3f}]", ha="center", va="bottom", fontsize=7.3, fontweight="bold", color="#1B2631")

ax_global.set_xticks(gx)
ax_global.set_xticklabels([board_titles.get(k, k) for k in board_keys])
ax_global.set_ylim(0.0, 128.0)
ax_global.set_ylabel("Match Outcomes (%)")
ax_global.set_title("(a) Outcome Shift & Global ΔScore (X - O)")
ax_global.legend(loc="upper center", ncol=3, frameon=True, fontsize=7.2)

slot_display = {
    "L1_Random": "L1 Random",
    "L2_RuleBased": "L2 RuleBased",
    "L3_AlphaBeta": "L3 AlphaBeta",
    "L4_MCTS": "L4 MCTS",
    "L5_StrongPolicy": "L5 Policy",
    "L6_RL": "L6 RL (Q/DQN)",
    "L7_HybridMCTS": "L7 HybridMCTS",
}
slots_set: set[str] = set()
slot_data: dict[str, dict[str, Any]] = {k: {} for k in board_keys}
for k in board_keys:
    for aid, m in boards_rq2[k]["agent_metrics"].items():
        slot = "L6_RL" if aid.startswith("L6_") else aid
        slots_set.add(slot)
        slot_data[k][slot] = m

ordered_slots = sorted(slots_set, key=_agent_order_key)
sx = np.arange(len(ordered_slots), dtype=np.float64)
bw4 = 0.25
for idx, k in enumerate(board_keys):
    vals, lows, highs, pos = [], [], [], []
    offset = (idx - (len(board_keys) - 1) / 2.0) * bw4
    for pos_i, slot in enumerate(ordered_slots):
        if slot in slot_data[k]:
            m = slot_data[k][slot]
            vals.append(float(m["delta_score"]))
            lows.append(float(m["delta_score_ci"][0]))
            highs.append(float(m["delta_score_ci"][1]))
            pos.append(sx[pos_i] + offset)
    ax_levels.bar(pos, vals, width=bw4, color=board_colors[k], alpha=0.88, edgecolor="#1B2631", lw=0.7, yerr=_ci_errors(vals, lows, highs), capsize=2.5, label=board_titles[k], zorder=3)

ax_levels.axhline(0.0, color="#222222", lw=1.0, ls="--", zorder=2)
ax_levels.set_xticks(sx)
ax_levels.set_xticklabels([slot_display.get(s, s) for s in ordered_slots], rotation=22, ha="right")
ax_levels.set_ylabel("ΔScore = Score(X) - Score(O) [95% CI]")
ax_levels.set_title("(b) First-Mover Advantage by Agent Level & Win-Length K")
ax_levels.set_ylim(-0.15, 0.92)
ax_levels.grid(axis="y", alpha=0.25, ls="--")
ax_levels.legend(loc="upper right", frameon=True)

fig4.suptitle("Fig 4: Ablation RQ2 — First-Mover Advantage Across Rule Variants & AI Levels", fontsize=12.5, fontweight="bold")
plt.show()

C:\Users\Ron\AppData\Local\Temp\ipykernel_28484\642663327.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Thảo luận Kết quả RQ2 — Sự sụp đổ của Lợi thế Đi trước từ `4×4 K=3` sang `4×4 K=4`

1. **Bùng nổ lợi thế đi trước trên `4×4, K=3`:**
   * Khi mở rộng kích thước lên 16 ô nhưng giữ chuỗi thắng ngắn `K=3`, tỷ lệ hòa sụp xuống **`0.0%`** (`400/400` ván đều phân định thắng thua), **`X Win = 76.5%`**, **`O Win = 23.5%`**, kéo **`ΔScore = +0.530 [+0.445, +0.610]`**.
   * Các tác tử mạnh như `L2_RuleBased`, `L3_AlphaBeta`, và `L6_DQN` đều đạt tuyệt đối **`Score(X) = 1.000`** khi được cầm quân `X` (đi trước), tạo thế song song / fork ngay từ những nước đầu mà `O` không thể cứu vãn.
2. **Sự sụp đổ của $\Delta\text{Score}$ trên chuẩn chính `4×4, K=4`:**
   * Khi nâng độ dài chuỗi thắng lên `K=4`, người đi sau có thêm không gian và thời gian để chặn đứng các đường mở (`10` đường thắng thay vì `24` đường trên `4×4 K=3`), đẩy tỷ lệ hòa lên **`74.1%`** (`X Win = 14.2%`, `O Win = 11.8%`).
   * Chênh lệch đi trước sụp đổ từ **`+0.530`** xuống chỉ còn **`+0.024 [-0.011, +0.058]`** (khoảng tin cậy 95% chứa `0`, tức gần như cân bằng hoàn toàn), thấp hơn rõ rệt cả cấu hình đối chứng cổ điển `3×3, K=3` (`Draw = 63.6%`, **`ΔScore = +0.112 [+0.073, +0.152]`**).
   * Điều này chứng minh **`4×4, K=4`** là chuẩn benchmark lý tưởng: loại bỏ thiên lệch lượt đi đầu và đo lường công bằng năng lực chiến thuật dài hạn của AI.

In [15]:
# Tái hiện Fig 5: RQ3 Learning Curves & Khả năng Tổng quát hóa trước Unseen Opponents
def _read_sl_curve(path: Path) -> tuple[list[int], list[float], list[float], list[float]]:
    ep, tr, va, acc = [], [], [], []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            obj = json.loads(line)
            if obj.get("type") == "epoch":
                ep.append(int(obj["epoch"]))
                tr.append(float(obj["train_loss"]))
                va.append(float(obj["val_loss"]))
                acc.append(float(obj["val_accuracy"]) if obj.get("val_accuracy") is not None else np.nan)
    return ep, tr, va, acc


def _read_rl_curves(paths: list[Path]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    per_seed = []
    for path in paths:
        rows, total_eps = [], 1
        with path.open("r", encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                obj = json.loads(line)
                if obj.get("type") == "meta":
                    total_eps = max(1, int(obj.get("episodes", 1)))
                elif obj.get("type") == "eval":
                    rows.append((int(obj["episode"]), float(obj["eval_score_vs_random"]), float(obj["eval_score_vs_rule"])))
        max_ep = max(total_eps, rows[-1][0])
        per_seed.append([(100.0 * ep / max_ep, sr, su) for ep, sr, su in rows])
    min_len = min(len(s) for s in per_seed)
    arr = np.asarray([s[:min_len] for s in per_seed], dtype=np.float64)
    return arr[0, :, 0], np.mean(arr[:, :, 1], axis=0), np.std(arr[:, :, 1], axis=0), np.mean(arr[:, :, 2], axis=0), np.std(arr[:, :, 2], axis=0)


def _plot_rq3_panel(ax: plt.Axes, board_data: dict[str, Any], title: str) -> None:
    cands = board_data["candidates"]
    sl_keys = [k for k in ("SL_FastPolicy", "SL_StrongPolicy", "SL_ValueGreedy") if k in cands]
    rl_keys = sorted(k for k in cands if k.startswith("RL_"))
    eval_axes = (
        ("vs_unseen_mcts", "Unseen MCTS"),
        ("vs_unseen_all", "Unseen All\n(MCTS + AB)"),
        ("vs_baseline", "Baseline\n(Rand + Rule)"),
        ("overall", "Overall\n(All Opps)"),
    )
    sl_styles = {
        "SL_FastPolicy": ("SL FastPolicy", PALETTE["sky"]),
        "SL_StrongPolicy": ("SL StrongPolicy", PALETTE["purple"]),
        "SL_ValueGreedy": ("SL ValueGreedy", PALETTE["amber"]),
    }
    series = []
    for sl_id in sl_keys:
        m = cands[sl_id]
        vals, lows, highs = [], [], []
        for mk, _ in eval_axes:
            src = m if mk == "overall" else m[mk]
            vals.append(float(src["score"]))
            lows.append(float(src["score_ci"][0]))
            highs.append(float(src["score_ci"][1]))
        lbl, col = sl_styles[sl_id]
        series.append((lbl, col, vals, lows, highs))

    rl_vals, rl_lows, rl_highs = [], [], []
    for mk, _ in eval_axes:
        s_scores = [float(cands[rk]["score"] if mk == "overall" else cands[rk][mk]["score"]) for rk in rl_keys]
        mu, sd = float(np.mean(s_scores)), float(np.std(s_scores))
        rl_vals.append(mu)
        rl_lows.append(max(0.0, mu - sd))
        rl_highs.append(min(1.0, mu + sd))
    rl_lbl = f"RL DQN ({len(rl_keys)}-seed μ±σ)" if any("DQN" in k for k in rl_keys) else f"RL TabQ ({len(rl_keys)}-seed μ±σ)"
    series.append((rl_lbl, PALETTE["green"], rl_vals, rl_lows, rl_highs))

    gx5 = np.arange(len(eval_axes), dtype=np.float64)
    bw5 = min(0.20, 0.78 / len(series))
    for idx, (lbl, col, vals, lows, highs) in enumerate(series):
        bars = ax.bar(gx5 + (idx - (len(series) - 1) / 2.0) * bw5, vals, width=bw5, color=col, alpha=0.90, edgecolor="#1B2631", lw=0.7, yerr=_ci_errors(vals, lows, highs), capsize=2.5, label=lbl, zorder=3)
        for b, v, hi in zip(bars, vals, highs):
            ax.text(b.get_x() + b.get_width() / 2.0, max(v, hi) + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=6.8, rotation=90)

    ax.axhline(0.50, color="#666666", ls="--", lw=0.9, zorder=2)
    ax.set_xticks(gx5)
    ax.set_xticklabels([lbl for _, lbl in eval_axes])
    ax.set_ylim(0.0, 0.96)
    ax.set_ylabel("Match Score = (W + 0.5·D) / N")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25, ls="--")
    ax.legend(loc="upper left", frameon=True, fontsize=7.5, ncol=2)


boards_rq3 = rq3_res.summary["boards"]
fig5, axes5 = plt.subplots(2, 2, figsize=(14.2, 9.6), constrained_layout=True)
ax_sl, ax_rl, ax_4x4, ax_3x3 = axes5[0, 0], axes5[0, 1], axes5[1, 0], axes5[1, 1]

ax_acc = ax_sl.twinx()
ax_acc.spines["top"].set_visible(False)
for fname, label, col in (
    ("train_strongpolicynet_4x4_k4.jsonl", "StrongPolicyNet", PALETTE["purple"]),
    ("train_fastpolicynet_4x4_k4.jsonl", "FastPolicyNet", PALETTE["blue"]),
    ("train_valuenet_4x4_k4.jsonl", "ValueNet (MSE)", PALETTE["amber"]),
):
    ep, tr, va, acc = _read_sl_curve(LOGS_DIR / fname)
    ax_sl.plot(ep, va, color=col, lw=2.0, label=f"{label} Val Loss")
    ax_sl.plot(ep, tr, color=col, lw=1.2, ls="--", alpha=0.65)
    valid_acc = [(e, a * 100.0) for e, a in zip(ep, acc) if not np.isnan(a)]
    if valid_acc:
        ax_acc.plot([p[0] for p in valid_acc], [p[1] for p in valid_acc], color=col, ls=":", lw=1.7, label=f"{label} Val Acc (%)")

ax_sl.set_xlabel("Training Epoch")
ax_sl.set_ylabel("Loss (Solid: Val | Dashed: Train)")
ax_acc.set_ylabel("Validation Accuracy (%, Dotted)")
ax_acc.set_ylim(25.0, 95.0)
ax_sl.grid(alpha=0.25, ls="--")
h1, l1 = ax_sl.get_legend_handles_labels()
h2, l2 = ax_acc.get_legend_handles_labels()
ax_sl.legend(h1 + h2, l1 + l2, loc="center right", frameon=True, fontsize=7.0)
ax_sl.set_title("(a) Supervised Learning Curves (4x4 K=4)")

pct_d, r_mu_d, r_sd_d, u_mu_d, u_sd_d = _read_rl_curves([LOGS_DIR / f"train_dqn_4x4_k4_seed{s}.jsonl" for s in (42, 123, 999)])
pct_t, r_mu_t, r_sd_t, _, _ = _read_rl_curves([LOGS_DIR / f"train_tabular_q_3x3_k3_seed{s}.jsonl" for s in (42, 123, 999)])

ax_rl.plot(pct_d, r_mu_d, color=PALETTE["green"], lw=2.0, marker="o", ms=4, label="DQN (4x4 K=4) vs Random (3-seed)")
ax_rl.fill_between(pct_d, r_mu_d - r_sd_d, np.minimum(1.0, r_mu_d + r_sd_d), color=PALETTE["green"], alpha=0.18)
ax_rl.plot(pct_d, u_mu_d, color=PALETTE["blue"], lw=1.8, ls="--", marker="s", ms=3.5, label="DQN (4x4 K=4) vs RuleBased")
ax_rl.fill_between(pct_d, u_mu_d - u_sd_d, u_mu_d + u_sd_d, color=PALETTE["blue"], alpha=0.14)
ax_rl.plot(pct_t, r_mu_t, color=PALETTE["vermilion"], lw=2.0, marker="^", ms=4, label="TabularQ (3x3 K=3) vs Random (3-seed)")
ax_rl.fill_between(pct_t, r_mu_t - r_sd_t, np.minimum(1.0, r_mu_t + r_sd_t), color=PALETTE["vermilion"], alpha=0.16)
ax_rl.axhline(0.50, color="#666666", ls=":", lw=1.0)
ax_rl.set_xlabel("Training Progress (% Episodes: 20k DQN / 60k TabQ)")
ax_rl.set_ylabel("Evaluation Score (Mean ± Std)")
ax_rl.set_ylim(0.38, 1.03)
ax_rl.grid(alpha=0.25, ls="--")
ax_rl.legend(loc="lower right", frameon=True, fontsize=7.4)
ax_rl.set_title("(b) RL Multi-Seed Learning Curves (Seeds 42, 123, 999)")

_plot_rq3_panel(ax_4x4, boards_rq3["4x4_k4"], "(c) RQ3 Generalization on 4x4 K=4 (SL vs 3-Seed DQN)")
_plot_rq3_panel(ax_3x3, boards_rq3["3x3_k3"], "(d) RQ3 Generalization on 3x3 K=3 (SL vs 3-Seed TabularQ)")

fig5.suptitle("Fig 5: Ablation RQ3 — Learning Curves & Generalization Against Unseen Opponents", fontsize=13.0, fontweight="bold")
plt.show()

C:\Users\Ron\AppData\Local\Temp\ipykernel_28484\595163921.py:140: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Thảo luận Kết quả RQ3 — Độ ổn định Đa Seed & Tổng quát hóa trước Unseen Opponents

1. **Nghịch lý "Validation Accuracy cao nhưng sụp đổ trước Unseen Tree Search" của Supervised Learning:**
   * Dù `StrongPolicyNet` đạt `80.21%` độ khớp nhãn chuyên gia trên tập validation (`split_policy = game_id_70_15_15`), khi đưa vào thực chiến trên `4×4, K=4` (`ablation_rq3_rl_generalization.jsonl`), điểm tổng thể chỉ còn **`0.497`** (`30` ván thua/360) và tụt mạnh xuống **`0.344`** khi gặp đối thủ tìm kiếm cây chưa từng thấy lúc tạo dữ liệu là **Unseen MCTS** (`ValueGreedy` thậm chí tụt xuống **`0.307`** tổng thể và **`0.256`** trước `MCTS`).
   * Nguyên nhân: Dữ liệu Supervised sinh từ `AlphaBeta` + `RuleBased` nên mạng học bị lệch phân phối (*distribution shift*) khi gặp các thế cờ trung cuộc biến dị từ `MCTS`, đồng thời không có cơ chế nhìn xa sửa sai chiến thuật.
2. **Sức mạnh tổng quát hóa & độ ổn định đa seed của RL Self-Play (`DQN` & `TabularQ`):**
   * Nhờ huấn luyện với **Mixed Pool (`70% Self-Play / 30% Baseline`)** qua 3 seed độc lập (`42, 123, 999`), **`DQNAgent` trên `4×4, K=4`** đạt điểm tổng thể **`0.586 ± 0.027`** (seed 42: `0.582`, seed 123: `0.621`, seed 999: `0.556` — chỉ `2–3` ván thua / 360 trận) và giữ vững **`0.554 ± 0.016`** trước **Unseen MCTS**, vượt trội toàn diện nhóm Supervised.
   * Trên bàn đối chứng **`3×3, K=3`**, **`TabularQAgent`** qua 3 seed đạt độ ổn định cực cao **`0.659 ± 0.005`** (`0.619 ± 0.023` trước `Unseen MCTS`, chỉ `0–1` ván thua / 360 trận) so với `0.469–0.485` của nhóm Supervised.

---
# PHẦN 6: TỔNG KẾT & KẾT LUẬN

## Bảng Tổng hợp Đối chiếu Rubric (`100/100` điểm), Hạn chế & Hướng Phát triển

### 6.1. Bảng Đối chiếu Tự Đánh giá Rubric Chính thức

| STT | Tiêu chí Rubric | Điểm tối đa | Minh chứng Thực nghiệm & Mã nguồn Đã Kiểm chứng | Đạt được |
|---:|---|---:|---|---|
| **1** | **Xác định bài toán / Game** | **10** | Mô hình hóa MDP đầy đủ cho `M×N, K`, 3 câu hỏi nghiên cứu `RQ1–RQ3`, kiểm chứng unit tests `test_engine.py`, `test_rules.py`, `test_config.py`. | **10 / 10** |
| **2** | **Xây dựng hệ thống** | **15** | Kiến trúc 5 phân hệ (`engine`, `agents`, `training`, `benchmark`, `ui`); 7+1 Game API (`clone()`, `undo()` $O(1)$), kiểm tra AST chống chạm private state, Rich CLI Dashboard. | **15 / 15** |
| **3** | **Tác tử tìm kiếm (Cấp 1–4)** | **20** | `Random`, `RuleBased` (6 tầng Newell & Simon), `Minimax / AlphaBeta` (cắt tỉa giảm `-94.71%` nút ở `d=4` + Iterative Deepening), `MCTS UCT` 4 pha chuẩn xác. | **20 / 20** |
| **4** | **Tác tử học được (Cấp 5–7)** | **15** | **15 checkpoints** đã huấn luyện thật + `*.meta.json` (SHA-256); chia tập theo `game_id`; `Fast/Strong PolicyNet`, `ValueNet`, `TabularQ` (3 seed), `Double-DQN Self-Play` (3 seed), `HybridMCTSAgent`. | **15 / 15** |
| **5** | **Thiết kế đánh giá** | **15** | **4 giải Round-Robin = 16,800 ván** (`200` ván/cặp, 50/50 `X/O`, chế độ `default`, `fixed_10ms`, `fixed_50ms`) + **4 kịch bản Ablation = 8,756 ván** (tổng `25,640` ván logged). | **15 / 15** |
| **6** | **Kết quả & thảo luận** | **10** | Đầy đủ Wilson/Bootstrap 95% CI, $\Delta\text{Score}$, `mean/p95_ms`, `avg_nodes/sims`, phân tích lỗi `sample_failure_cases.md` và trả lời định lượng trọn vẹn `RQ1–RQ3`. | **10 / 10** |
| **7** | **Báo cáo, slide, demo** | **10** | Pipeline tự động xuất `Fig 1..5` (300 DPI), Demo Rich CLI (`Human vs AI`, `AI vs AI`), bộ 6 tài liệu `Planning/` và Notebook tương tác `Extended_Tic_Tac_Toe.ipynb`. | **10 / 10** |
| **8** | **Tái lập & đạo đức** | **5** | `347/347 pytest` pass, `ruff check` sạch PEP 8, `seed_everything()`, tái lập 1 lệnh qua `scripts/`, minh bạch ranh giới kế thừa `References/` (*Zero Fabrication*). | **5 / 5** |
| **+** | **Điểm Khuyến khích (Bonus)** | **+10** | Thực nghiệm thuyết phục **Cấp 7 `HybridMCTS`** (`rollout` & `mixed` + LRU Cache), kiểm chứng đa bàn (`3×3`, `4×4 K=3/4`, `5×5 K=4`), và **Action Heatmap 5 mức màu** (`Prior` vs `Visit Share`). | **+10 / +10** |
| | **TỔNG ĐIỂM MỤC TIÊU** |  | **Toàn bộ hệ thống đạt chuẩn hoàn thiện & sẵn sàng nghiệm thu.** |  |

---

### 6.2. Hạn chế Hiện tại
1. **Chi phí suy luận PyTorch đơn lẻ trên CPU trong `HybridMCTSAgent`:** Mỗi lần mở rộng nút lá phải gọi `StrongPolicyNet`/`ValueNet` với batch size `1` trên CPU, làm giảm thông lượng từ `190.7 sims` (`PureMCTS`) xuống `145.9 sims` (`HybridMixed`) trong giới hạn `10 ms/move` (dù LRU Cache đã cắt giảm `47%–69%` số lượt gọi mạng).
2. **Trần năng lực của Supervised Learning:** Mạng Policy/Value huấn luyện từ dữ liệu `AlphaBeta` + `RuleBased` vẫn bị giới hạn bởi chất lượng dữ liệu teacher và sụt giảm điểm khi gặp đối thủ cây MCTS ngoài phân phối huấn luyện.
3. **Chưa có vòng lặp Self-Play RL tích hợp cây PUCT (AlphaZero Loop):** `HybridMCTSAgent` hiện tách rời bước huấn luyện mạng (Supervised) và bước tìm kiếm cây.

### 6.3. Hướng Mở rộng & Phát triển Tương lai
1. **Nâng cấp lên kiến trúc AlphaZero hoàn chỉnh:** Hợp nhất thành mạng **Dual-Head ResNet** (`Policy Head + Value Head`), huấn luyện bằng chính phân phối lượt thăm MCTS $\pi_t \propto N(s_t, a)^{1/\tau}$ qua vòng lặp *Self-Play → Replay Buffer → Update* kết hợp **Dirichlet Noise** tại nút gốc và **Virtual Loss Batched Inference** trên GPU.
2. **Mở rộng sang Gomoku (`15×15, K=5`):** Áp dụng kiến trúc Engine `M×N, K` hiện có kèm luật cân bằng lượt đi đầu (**Swap2 / Renju**) để triệt tiêu lợi thế đi trước, kết hợp **Threat-Space Search (VCF/VCT)** và mạng **Fully Convolutional** để mở rộng quy mô lên $3^{225}$ trạng thái.